# CASMI26 submission
Ported prvsiyan 4-channel + HistGBM pipeline (`src/casmi` in the repo).

In [ ]:
!mkdir -p /kaggle/working/casmi

In [ ]:
%%writefile /kaggle/working/casmi/__init__.py
"""CASMI 2026 retrieval pipeline (port of prvsiyan analog-propagation baseline)."""


In [ ]:
%%writefile /kaggle/working/casmi/candidates.py
"""Candidate pool = COCONUT (attached, precomputed fps) U train structures (fingerprinted here), InChIKey14-deduped,
sorted by exact mass. Optional ChEBI+LIPID MAPS (CFG.USE_BIO_DB, off by default as in the notebook).

Cache: data/cache/pool_train_fp.npy / pool_train_meta.parquet (train-structure half, ~5 min once).
"""
from __future__ import annotations

import multiprocessing as mp
import pickle
import time

import numpy as np
import polars as pl

from . import chem, config

C = config.CACHE
_BITS = None


def _init_bits(bits_path):
    global _BITS
    _BITS = np.load(bits_path)


def _fp_and_mass(smi):
    r = chem.raw_fp(smi)
    if r is None:
        return None
    return r[0][_BITS], r[1]


def build_train_fp_cache(cfg: config.CFG, force=False):
    fpp, mp_ = C / 'pool_train_fp.npy', C / 'pool_train_meta.parquet'
    if fpp.exists() and mp_.exists() and not force:
        print('train-structure fingerprint cache exists'); return
    t0 = time.time()
    cm = pickle.load(open(config.COCO_DIR / 'coco_meta.pkl', 'rb'))
    co = [str(k) for k in cm['keys']]
    st = pl.read_parquet(C / 'structs.parquet')
    tr = st.filter(~pl.col('inchikey14').is_in(co))
    smis = tr['normalized_smiles'].to_list()
    print(f'fingerprinting {len(smis):,} training structures not in COCONUT', flush=True)
    ctx = mp.get_context('spawn')
    with ctx.Pool(cfg.WORKERS, initializer=_init_bits, initargs=(str(config.COCO_DIR / 'fp_bits.npy'),)) as p:
        res = p.map(_fp_and_mass, smis, chunksize=500)
    ok = [i for i, r in enumerate(res) if r is not None]
    fp = np.packbits(np.stack([res[i][0] for i in ok]), axis=1)
    mass = np.array([res[i][1] for i in ok])
    np.save(fpp, fp)
    pl.DataFrame(dict(key=tr['inchikey14'].to_numpy()[ok], smiles=np.asarray(smis, dtype=object)[ok].tolist(),
                      mass=mass)).write_parquet(mp_)
    print(f'train fp cache: {len(ok):,} ok / {len(smis):,} ({time.time()-t0:.0f}s)', flush=True)


class Pool:
    """Candidates + packed fingerprints, sorted by exact mass."""

    def __init__(self, cfg: config.CFG):
        t0 = time.time()
        cm = pickle.load(open(config.COCO_DIR / 'coco_meta.pkl', 'rb'))
        fp = [np.load(config.COCO_DIR / 'coco_fp.npy')]
        mass = [np.load(config.COCO_DIR / 'coco_mass.npy')]
        keys = [np.asarray(cm['keys'], dtype=object)]
        smis = [np.asarray(cm['smiles'], dtype=object)]
        src = [np.zeros(len(mass[0]), np.int8)]
        if cfg.USE_BIO_DB:
            bm = pickle.load(open(config.BIO_DIR / 'bio_meta.pkl', 'rb'))
            fp.append(np.load(config.BIO_DIR / 'bio_fp.npy')); mass.append(np.load(config.BIO_DIR / 'bio_mass.npy'))
            keys.append(np.asarray(bm['keys'], dtype=object)); smis.append(np.asarray(bm['smiles'], dtype=object))
            src.append(np.full(len(mass[-1]), 2, np.int8))
        tm = pl.read_parquet(C / 'pool_train_meta.parquet')
        fp.append(np.load(C / 'pool_train_fp.npy')); mass.append(tm['mass'].to_numpy())
        keys.append(tm['key'].to_numpy().astype(object)); smis.append(tm['smiles'].to_numpy().astype(object))
        src.append(np.ones(len(tm), np.int8))
        fp = np.vstack(fp); mass = np.concatenate(mass); keys = np.concatenate(keys)
        smis = np.concatenate(smis); src = np.concatenate(src)
        good = np.isfinite(mass)
        fp, mass, keys, smis, src = fp[good], mass[good], keys[good], smis[good], src[good]
        o = np.argsort(mass)
        self._fp = fp[o]; self.mass = mass[o]; self.keys = keys[o]; self.smiles = smis[o]
        self.src = src[o]  # provenance: NEVER a ranker feature (leaks in the class-2 simulation)
        self.nbits = cm['nbits']
        self.k2i = {k: i for i, k in enumerate(self.keys)}
        print(f'pool: {len(self.mass):,} structures, {self.nbits} bits ({time.time()-t0:.0f}s)', flush=True)

    def window(self, t, ppm):
        a = np.searchsorted(self.mass, t * (1 - ppm / 1e6), 'left')
        b = np.searchsorted(self.mass, t * (1 + ppm / 1e6), 'right')
        return np.arange(a, b)

    def fps(self, idx):
        return np.unpackbits(np.asarray(self._fp[idx]), axis=1)[:, :self.nbits]


In [ ]:
%%writefile /kaggle/working/casmi/chem.py
"""Masses, adducts, neutral-mass derivation and RDKit fingerprints (identical constants to the notebook)."""
from __future__ import annotations

import re

import numpy as np

MASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,
            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,
            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)
E = 0.00054857990
PROTON = MASS['H'] - E
H2O = 2 * MASS['H'] + MASS['O']
NH4 = MASS['N'] + 4 * MASS['H']
FORMATE = MASS['C'] + 2 * MASS['H'] + 2 * MASS['O']
ACETATE = 2 * MASS['C'] + 4 * MASS['H'] + 2 * MASS['O']
ADDUCTS = {
    "[M+H]+": (1, 1, PROTON), "[M+NH4]+": (1, 1, NH4 - E), "[M+Na]+": (1, 1, MASS['Na'] - E),
    "[M+K]+": (1, 1, MASS['K'] - E), "[M-H2O+H]+": (1, 1, PROTON - H2O), "[M-2H2O+H]+": (1, 1, PROTON - 2 * H2O),
    "[M+2H]2+": (1, 2, 2 * PROTON), "[M]+": (1, 1, -E), "[M-H2O]+": (1, 1, -E - H2O),
    "[M+CH3OH+H]+": (1, 1, PROTON + MASS['C'] + 4 * MASS['H'] + MASS['O']),
    "[M+CH3CN+H]+": (1, 1, PROTON + 2 * MASS['C'] + 3 * MASS['H'] + MASS['N']),
    "[M-H]-": (1, 1, -PROTON), "[M-H2O-H]-": (1, 1, -PROTON - H2O), "[M+CH2O2-H]-": (1, 1, FORMATE - PROTON),
    "[M+C2H4O2-H]-": (1, 1, ACETATE - PROTON), "[M+Cl]-": (1, 1, MASS['Cl'] + E), "[M]-": (1, 1, E),
    "[M-2H]-": (1, 2, -2 * PROTON), "[M+Na-2H]-": (1, 1, MASS['Na'] - 2 * PROTON),
    "[2M+H]+": (2, 1, PROTON), "[2M+Na]+": (2, 1, MASS['Na'] - E), "[2M+NH4]+": (2, 1, NH4 - E),
    "[2M+K]+": (2, 1, MASS['K'] - E), "[2M-H]-": (2, 1, -PROTON), "[2M+CH2O2-H]-": (2, 1, FORMATE - PROTON),
    "[2M+C2H4O2-H]-": (2, 1, ACETATE - PROTON), "[2M+Na-2H]-": (2, 1, MASS['Na'] - 2 * PROTON),
    "[3M+H]+": (3, 1, PROTON), "[3M-H]-": (3, 1, -PROTON),
}
MONO = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, S=31.97207100,
            P=30.97376163, Cl=34.96885268, Br=78.9183371, F=18.99840322, I=126.904473,
            Si=27.9769265325, Se=79.9165213, Na=22.9897692809, K=38.96370668, B=11.0093054,
            As=74.9215965, Fe=55.9349375, D=2.0141017778)
_TOK = re.compile(r'([A-Z][a-z]?)(\d*)')


def formula_mass(f):
    """Neutral monoisotopic mass by arithmetic on a molecular formula (NaN if unknown element)."""
    if not isinstance(f, str) or not f:
        return np.nan
    m = 0.0
    for el, n in _TOK.findall(f):
        if not el:
            continue
        if el not in MONO:
            return np.nan
        m += MONO[el] * (int(n) if n else 1)
    return m


def neutral_mass(mz, adduct):
    mz = np.asarray(mz, np.float64)
    out = np.full(len(mz), np.nan)
    ad = np.asarray(adduct, dtype=object)
    for a, (n, z, d) in ADDUCTS.items():
        m = (ad == a)
        if m.any():
            out[m] = (mz[m] * z - d) / n
    return out


# ---------------------------------------------------------------- fingerprints (ECFP4|ECFP6|RDKit|MACCS)[BITS]
_g: dict = {}


def _fp_init():
    from rdkit import RDLogger
    from rdkit.Chem import rdFingerprintGenerator
    RDLogger.DisableLog('rdApp.*')
    _g['m2'] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
    _g['m3'] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
    _g['rk'] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)


def raw_fp(smi):
    """Full 10,407-bit fingerprint (uint8) and exact mass, or None."""
    from rdkit import Chem
    from rdkit.Chem import MACCSkeys
    from rdkit.Chem.Descriptors import ExactMolWt
    if not _g:
        _fp_init()
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return None
    try:
        fp = np.concatenate([_g['m2'].GetFingerprintAsNumPy(m).astype(np.uint8),
                             _g['m3'].GetFingerprintAsNumPy(m).astype(np.uint8),
                             _g['rk'].GetFingerprintAsNumPy(m).astype(np.uint8),
                             np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])
        return fp, float(ExactMolWt(m))
    except Exception:
        return None


In [ ]:
%%writefile /kaggle/working/casmi/cli.py
"""CLI:  uv run casmi build | cv | predict

  casmi build                               # one-time caches (library, metric keys, train-structure fingerprints)
  casmi cv --n-np 200 --n-e180 300 --tag exp000
  casmi predict --ranker rank_train --out outputs/submission.csv
  casmi predict --ranker cv --cv-tag exp000 --out outputs/submission_cvranker.csv
  casmi predict --ranker v2 --ranker-rows outputs/ranker_rows_v2 --fp-dir data/ext/casmi26-fp-models-h1
"""
from __future__ import annotations

import argparse
import glob
import json
import time

from . import config


def _engine(cfg, prefer_instrument='timsTOF'):
    from .candidates import Pool
    from .engine import Engine
    from .frag import Fragmenter
    from .library import Library
    bank = None
    if cfg.USE_FP_MODEL:
        from .fpnet import ModelBank
        paths = sorted(glob.glob(str(config.FP_MODEL_DIR / 'fp_*.pt')))
        bank = ModelBank(paths, device=cfg.DEVICE) if paths else None
    L = Library(cfg)
    P = Pool(cfg)
    fr = Fragmenter(cfg.WORKERS) if cfg.USE_FRAG else None
    return Engine(cfg, L, P, bank, fr, prefer_instrument)


def main(argv=None):
    ap = argparse.ArgumentParser('casmi')
    sub = ap.add_subparsers(dest='cmd', required=True)
    sub.add_parser('build')
    c = sub.add_parser('cv')
    c.add_argument('--n-np', type=int, default=200); c.add_argument('--n-e180', type=int, default=300)
    c.add_argument('--seed', type=int, default=0); c.add_argument('--tag', default='exp000')
    c.add_argument('--no-reuse', action='store_true')
    c.add_argument('--holdout', choices=['cv', 'all'], default=None,
                   help='fixed list from holdout_v1.json (cv == exp000 molecules) instead of sampling')
    p = sub.add_parser('predict')
    p.add_argument('--ranker', choices=['rank_train', 'cv', 'v2'], default='rank_train')
    p.add_argument('--ranker-rows', default=None, help='v2: rows dir from `python -m casmi.rows_v2 build`')
    p.add_argument('--v2-model', choices=['blend', 'hgb', 'lgb'], default='blend', help='v2: ranker kind')
    p.add_argument('--v2-cols', choices=['all', 'base'], default='all', help='v2: base = 31 cols, all = + prior + FP-family')
    p.add_argument('--cv-tag', default='exp000')
    p.add_argument('--out', default=str(config.OUTPUTS / 'submission.csv'))
    for s in (c, p):
        s.add_argument('--device', default='cpu'); s.add_argument('--no-fp', action='store_true')
        s.add_argument('--no-frag', action='store_true'); s.add_argument('--workers', type=int, default=None)
        s.add_argument('--fp-dir', default=None, help='directory with fp_*.pt (default: data/ext/casmi26-fp-models-v2)')
    a = ap.parse_args(argv)
    cfg = config.CFG()
    if getattr(a, 'workers', None): cfg.WORKERS = a.workers
    if getattr(a, 'device', None): cfg.DEVICE = a.device
    if getattr(a, 'no_fp', False): cfg.USE_FP_MODEL = False
    if getattr(a, 'no_frag', False): cfg.USE_FRAG = False
    if getattr(a, 'fp_dir', None):
        from pathlib import Path
        config.FP_MODEL_DIR = Path(a.fp_dir)
    T0 = time.time()
    if a.cmd == 'build':
        from .candidates import build_train_fp_cache
        from .library import build_library_cache
        build_library_cache(cfg); build_train_fp_cache(cfg)
    elif a.cmd == 'cv':
        from .cv import run_cv
        E = _engine(cfg)
        print(f'engine ready {time.time()-T0:.0f}s', flush=True)
        run_cv(E, cfg, a.n_np, a.n_e180, a.seed, a.tag, reuse=not a.no_reuse, holdout=a.holdout)
    elif a.cmd == 'predict':
        from . import submit
        from .ranker import Ranker
        queries, pref = submit.test_queries()
        E = _engine(cfg, pref)
        t_eng = time.time() - T0
        if a.ranker == 'rank_train':
            rk = Ranker.from_rank_train(cfg)
        elif a.ranker == 'v2':
            from .rows_v2 import DEFAULT_DIR, RankerV2
            rk = RankerV2.from_rows(cfg, a.ranker_rows or DEFAULT_DIR, a.v2_model, a.v2_cols)
        else:
            from .cv import FP_COLS, cv_ranker
            rk = cv_ranker(cfg, a.cv_tag, FP_COLS if not cfg.USE_FP_MODEL else None)
        t1 = time.time()
        rows, diag = submit.predict_test(E, rk, queries, cfg)
        t_pred = time.time() - t1
        df = submit.write_submission(rows, a.out)
        meta = dict(out=a.out, ranker=a.ranker, secs_ranker_fit=round(t1 - T0 - t_eng),
                    **({'v2': dict(model=a.v2_model, cols=a.v2_cols, rows=str(a.ranker_rows))} if a.ranker == 'v2' else {}),
                    fp_dir=str(config.FP_MODEL_DIR), secs_total=round(time.time() - T0), secs_setup=round(t_eng),
                    secs_predict=round(t_pred), n=len(df), mean_len=float(df['smiles'].str.split(';').list.len().mean()))
        json.dump(meta, open(a.out.replace('.csv', '.meta.json'), 'w'), indent=1)
        print(f'wrote {a.out} {df.shape}; {meta}', flush=True)
    if getattr(locals().get('E', None), 'frag', None) is not None:
        E.frag.close()


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/casmi/config.py
"""Configuration. Defaults reproduce prvsiyan/analog-propagation-casmi-2026-baseline (the 0.335/0.328 config).

Every constant carries the notebook's own measurement comment in research/04-pipeline-analysis.md.
"""
from __future__ import annotations

import os
from dataclasses import dataclass, field
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
DATA = Path(os.environ.get("CASMI_DATA", ROOT / "data"))
EXT = Path(os.environ.get("CASMI_EXT", DATA / "ext"))
CACHE = Path(os.environ.get("CASMI_CACHE", DATA / "cache"))
OUTPUTS = Path(os.environ.get("CASMI_OUTPUTS", ROOT / "outputs"))

TRAIN = DATA / "train.parquet"
TEST = DATA / "test.parquet"
SAMPLE = DATA / "sample_submission.csv"

COCO_DIR = EXT / "coconut-casmi26-candidates"
BIO_DIR = EXT / "chebi-lipidmaps-casmi26"
FP_MODEL_DIR = EXT / "casmi26-fp-models-v2"
RANK_TRAIN = EXT / "casmi26-ranker-features" / "rank_train.npz"


@dataclass
class CFG:
    # candidate generation
    PPM_WIN: float = 10.0
    PPM_FALLBACK: float = 30.0
    # spectrum cleaning (library + query side of the similarity channels)
    INT_FLOOR: float = 0.002
    MAX_PEAKS: int = 256
    MZ_TOL: float = 0.01
    INT_POWER: float = 1.0
    ENT_WEIGHT: bool = True
    # analog propagation
    ANALOG_WIN: float = 200.0
    N_ANALOG: int = 200
    SIM_POWER: float = 3.0  # used inside rank_features (P_SIM)
    # ranker
    W1_PRIORS: tuple = (0.30, 0.60)
    SEEDS: tuple = (0, 1, 2, 3)
    GBM: dict = field(default_factory=lambda: dict(max_depth=6, max_iter=500, learning_rate=0.03,
                                                   min_samples_leaf=80, l2_regularization=1.0))
    USE_BIO_DB: bool = False
    CAND_CAP: int = 500
    LIB_MASS_FORMULA: bool = True
    ANALOG_MATCH_INSTR: bool = True
    TOPN: int = 25
    # channels on/off (for ablations)
    USE_FP_MODEL: bool = True
    USE_FRAG: bool = True
    # runtime
    WORKERS: int = max(1, (os.cpu_count() or 4) - 1)
    DEVICE: str = "cpu"  # 'cpu' | 'mps' | 'cuda' for FPNet


In [ ]:
%%writefile /kaggle/working/casmi/cv.py
"""Leak-free CV on timsTOF held-out molecules.

Held-out molecules = structures sampled from enveda-np-examples and enveda-180 (the only timsTOF libraries),
deduplicated by the METRIC key (tautomer-canonical InChIKey14). Each is queried with k of its own timsTOF spectra
(k drawn from the test's spectra-per-molecule distribution). The "target group" G = every library structure
(inchikey14) that shares the target's metric key, so tautomer/stereo duplicates cannot leak.

  split A (class-1-like): hide G's spectra in the query's source library only; other libraries keep theirs.
                          Only molecules whose G has spectra in another library are eligible.
  split B (class-2-like): hide ALL spectra of G (library search + analog representatives); G stays in the pool.
  split C (class-3-like): as B, and also remove every pool member with the target's metric key.

The ranker is retrained inside CV (5 folds by molecule; rows from A (M=0) + B (M=1) of the training folds).
rank_train.npz is NOT used for CV numbers: it stores no molecule identities (only 819 group ids) and was built from
a 250-NP + 569-obscure holdout, i.e. almost certainly contains enveda-np-examples. It is reported only as a
"leaky reference" row.

Known remaining leak: the attached FPNet weights (casmi26-fp-models-v2) were trained on train.parquet with an
unknown structure split, so held-out structures may have been seen by the fingerprint model. cv reports a no-FP
variant and the FP-alone MRR per source library so the size of that leak is visible.
"""
from __future__ import annotations

import json
import pickle
import time
from pathlib import Path

import numpy as np
import polars as pl

from . import config
from .engine import Mask, Query
from .features import FEATURE_NAMES
from .metric import score_key
from .ranker import Ranker

NP_LIB, E180_LIB = 'enveda-np-examples', 'enveda-180'
WEIGHTS = dict(A=0.16, B=0.45, C=0.39)
FP_COLS = list(range(15, 21)) + list(range(25, 31))


# --------------------------------------------------------------------------------------------- holdout
def select_holdout(L, n_np=200, n_e180=300, seed=0, exclude_mkeys=()):
    rng = np.random.default_rng(seed)
    te = pl.read_parquet(config.TEST, columns=['molecule_id'])
    k_dist = te.group_by('molecule_id').len()['len'].to_numpy()
    chosen, used_mkeys = [], set(exclude_mkeys)
    for lib, n in ((NP_LIB, n_np), (E180_LIB, n_e180)):
        sids = np.unique(L.sid[L.lib == lib])
        rng.shuffle(sids)
        got = 0
        for s in sids:
            mk = L.s_mkey[s]
            if mk is None or mk in used_mkeys:
                continue
            G = np.array(sorted(L.mkey2sids[mk]), np.int64)
            sp = np.concatenate([L.spectra_of(g) for g in G])
            q_sp = sp[(L.lib[sp] == lib) & np.isfinite(L.nm[sp])]
            if len(q_sp) == 0:
                continue
            k = int(min(len(q_sp), rng.choice(k_dist)))
            q_sp = np.sort(rng.choice(q_sp, size=k, replace=False))
            eligible_A = bool((L.lib[sp] != lib).any())
            used_mkeys.add(mk)
            chosen.append(dict(sid=int(s), mkey=mk, smiles=L.s_smiles[s], src_lib=lib, G=G.tolist(),
                               q_rows=q_sp.tolist(), eligible_A=eligible_A))
            got += 1
            if got >= n:
                break
    return chosen


def load_query_rows(rows):
    """Raw (uncleaned) spectra for library row indices, read from train.parquet (row index == library index)."""
    rows = sorted(set(rows))
    df = (pl.scan_parquet(config.TRAIN).with_row_index('row')
            .filter(pl.col('row').is_in(rows))
            .select('row', 'ms2_mzs', 'ms2_normalized_intensities', 'precursor_mz', 'adduct', 'instrument_type',
                    'collision_energy_ev', 'ionization_mode').collect())
    out = {}
    for r in df.iter_rows(named=True):
        out[r['row']] = dict(mz=np.asarray(r['ms2_mzs'], np.float64), it=np.asarray(r['ms2_normalized_intensities'], np.float64),
                             prec=float(r['precursor_mz']), adduct=r['adduct'], instrument=r['instrument_type'],
                             ce=r['collision_energy_ev'], mode=1.0 if r['ionization_mode'] == 'positive' else -1.0)
    return out


def make_query(L, h, raw):
    from .chem import neutral_mass
    specs = [raw[i] for i in h['q_rows']]
    nm = neutral_mass(np.array([s['prec'] for s in specs]), np.array([s['adduct'] for s in specs], dtype=object))
    nm = nm[np.isfinite(nm)]
    if not len(nm):
        return None
    return Query(specs, float(np.median(nm)))


# --------------------------------------------------------------------------------------------- features
def _pack(c, truth):
    if c is None:
        return None
    keys = [score_key(s) for s in c['smiles']]
    Y = np.array([k == truth for k in keys], np.float32)
    return dict(X=c['X'], Y=Y, keys=keys, n=len(keys))


def compute_features(E, holdout, raw, log_every=25):
    L = E.L
    out, t0 = [], time.time()
    tim = dict(logits=0.0, lib=0.0, analog=0.0, cand=0.0)
    for i, h in enumerate(holdout):
        q = make_query(L, h, raw)
        rec = dict(h=h, A=None, B=None, C=None)
        if q is not None:
            truth = h['mkey']; G = np.array(h['G'], np.int64)
            ta = time.time(); z = E.logits(q); tim['logits'] += time.time() - ta
            mB = Mask(excl_sids=G)
            ta = time.time(); libB = E.lib_sim(q, mB); tim['lib'] += time.time() - ta
            ta = time.time(); anB = E.analog_sim(q, mB); tim['analog'] += time.time() - ta
            ta = time.time()
            rec['B'] = _pack(E.candidates(q, libB, anB, z), truth)
            rec['C'] = _pack(E.candidates(q, libB, anB, z, pool_excl_mkeys=frozenset([truth])), truth)
            tim['cand'] += time.time() - ta
            if h['eligible_A']:
                mA = Mask(src_sids=G, src_lib=h['src_lib'])
                bad = lambda idx: mA.spectra_bad(L, idx)
                ov = {int(s): L.rep_excluding(int(s), bad) for s in G}
                ta = time.time(); libA = E.lib_sim(q, mA); tim['lib'] += time.time() - ta
                ta = time.time(); anA = E.analog_sim(q, mA, ov); tim['analog'] += time.time() - ta
                ta = time.time(); rec['A'] = _pack(E.candidates(q, libA, anA, z), truth); tim['cand'] += time.time() - ta
            rec['target'] = q.target; rec['n_spec'] = len(q.spectra)
            rec['lib_max_B'] = float(max(libB.values())) if libB else 0.0
        out.append(rec)
        if (i + 1) % log_every == 0 or i + 1 == len(holdout):
            el = time.time() - t0
            print(f'  cv feats {i+1}/{len(holdout)}  {el:.0f}s ({el/(i+1):.2f}s/mol)  '
                  + ' '.join(f'{k}={v:.0f}s' for k, v in tim.items()), flush=True)
    return out, tim


# --------------------------------------------------------------------------------------------- evaluation
def rr_from_scores(pack, score, topn=25):
    """Reciprocal rank after metric-key dedupe + invalid filtering (exactly what submit.select does)."""
    if pack is None:
        return 0.0
    order = np.argsort(-score, kind='stable')
    seen, r = set(), 0
    for i in order:
        k = pack['keys'][i]
        if k is None or k in seen:
            continue
        seen.add(k); r += 1
        if pack['Y'][i] > 0:
            return 1.0 / r
        if r >= topn:
            break
    return 0.0


def _stack(recs, split, cols_zero=None):
    X, Y, M = [], [], []
    for r in recs:
        p = r[split]
        if p is None or p['n'] == 0:
            continue
        x = p['X'].copy()
        if cols_zero:
            x[:, cols_zero] = 0.0
        X.append(x); Y.append(p['Y']); M.append(np.full(p['n'], 0 if split == 'A' else 1))
    if not X:
        return None
    return np.vstack(X), np.concatenate(Y), np.concatenate(M)


def evaluate(recs, cfg, n_folds=5, seed=0, variants=('full', 'noFP'), use_rank_train=True):
    rng = np.random.default_rng(seed)
    fold = rng.permutation(len(recs)) % n_folds
    res = {}
    t0 = time.time()
    for var in variants:
        cz = FP_COLS if var == 'noFP' else None
        rr = {s: np.full(len(recs), np.nan) for s in 'ABC'}
        for f in range(n_folds):
            tr = [recs[i] for i in range(len(recs)) if fold[i] != f]
            parts = [p for p in (_stack(tr, 'A', cz), _stack(tr, 'B', cz)) if p is not None]
            X = np.vstack([p[0] for p in parts]); Y = np.concatenate([p[1] for p in parts])
            M = np.concatenate([p[2] for p in parts])
            rk = Ranker(cfg).fit(X, Y, M)
            for i in np.where(fold == f)[0]:
                for s in 'ABC':
                    p = recs[i][s]
                    if s == 'A' and not recs[i]['h']['eligible_A']:
                        continue
                    if p is None:
                        rr[s][i] = 0.0; continue
                    x = p['X'].copy()
                    if cz: x[:, cz] = 0.0
                    rr[s][i] = rr_from_scores(p, rk.predict(x))
        res[var] = rr
        print(f'  ranker CV [{var}] done ({time.time()-t0:.0f}s)', flush=True)
    if use_rank_train and config.RANK_TRAIN.exists():
        rk = Ranker.from_rank_train(cfg)
        rr = {s: np.full(len(recs), np.nan) for s in 'ABC'}
        for i, r in enumerate(recs):
            for s in 'ABC':
                if s == 'A' and not r['h']['eligible_A']:
                    continue
                rr[s][i] = rr_from_scores(r[s], rk.predict(r[s]['X'])) if r[s] is not None else 0.0
        res['rank_train(leaky)'] = rr
    # single-channel baselines on split B (no ranker => no ranker leak); FP-alone exposes the FPNet leak
    single = {}
    for name, col in (('lib', 0), ('analog_ap', 5), ('fpnet_fz', 17), ('frag', 21)):
        v = np.full(len(recs), np.nan)
        for i, r in enumerate(recs):
            v[i] = rr_from_scores(r['B'], r['B']['X'][:, col]) if r['B'] is not None else 0.0
        single[name] = v
    return res, single, fold


def summarize(recs, res, single):
    libs = np.array([r['h']['src_lib'] for r in recs])
    lines = []

    def m(v, mask=None):
        v = v if mask is None else v[mask]
        v = v[~np.isnan(v)]
        return (float(v.mean()) if len(v) else float('nan')), len(v)

    table = {}
    for var, rr in res.items():
        row = {}
        for s in 'ABC':
            row[s] = m(rr[s])
            for lib in (NP_LIB, E180_LIB):
                row[f'{s}@{lib}'] = m(rr[s], libs == lib)
        row['total'] = sum(WEIGHTS[s] * row[s][0] for s in 'ABC')
        # per-source weighted totals (A taken from the same source when available, else the pooled A)
        for lib in (NP_LIB, E180_LIB):
            a = row[f'A@{lib}'][0] if row[f'A@{lib}'][1] >= 20 else row['A'][0]
            row[f'total@{lib}'] = WEIGHTS['A'] * a + WEIGHTS['B'] * row[f'B@{lib}'][0] + WEIGHTS['C'] * row[f'C@{lib}'][0]
        table[var] = row
        lines.append(f"{var:>18}: A={row['A'][0]:.4f} (n={row['A'][1]})  B={row['B'][0]:.4f} (n={row['B'][1]})  "
                     f"C={row['C'][0]:.4f}  weighted(16/45/39)={row['total']:.4f}  "
                     f"weighted[np]={row['total@' + NP_LIB]:.4f}  weighted[e180]={row['total@' + E180_LIB]:.4f}")
        lines.append(' ' * 20 + '  '.join(f"{s}[{'np' if lib == NP_LIB else 'e180'}]={row[f'{s}@{lib}'][0]:.4f}(n={row[f'{s}@{lib}'][1]})"
                                            for s in 'AB' for lib in (NP_LIB, E180_LIB)))
    lines.append('single-channel MRR on split B (no ranker):')
    for name, v in single.items():
        lines.append(f"   {name:>10}: all={m(v)[0]:.4f}  np={m(v, libs == NP_LIB)[0]:.4f}  e180={m(v, libs == E180_LIB)[0]:.4f}")
    recall = np.mean([r['B'] is not None and r['B']['Y'].any() for r in recs])
    ncand = np.median([r['B']['n'] for r in recs if r['B'] is not None])
    lines.append(f'split-B window recall={recall:.4f}  median candidates={ncand:.0f}  '
                 f"A-eligible={np.mean([r['h']['eligible_A'] for r in recs]):.3f}")
    return table, '\n'.join(lines)


def fixed_holdout(which='cv'):
    """Fixed held-out list shared with FPNet training (src/casmi/holdout_v1.json, see train_fpnet.make_holdout).
    'cv' = the exp000 molecules (250 np-examples + 250 enveda-180); 'all' = cv + the extra enveda-180 molecules.
    Every metric key in 'all' is excluded from the retrained FPNet (fp_*_h1.pt), so CV on it is FP-leak-free."""
    from .train_fpnet import load_holdout
    H = load_holdout()
    return H['cv'] + (H['extra'] if which == 'all' else [])


def run_cv(E, cfg, n_np=200, n_e180=300, seed=0, tag='exp000', reuse=True, holdout=None):
    outdir = config.OUTPUTS / 'cv' / tag
    outdir.mkdir(parents=True, exist_ok=True)
    fpath = outdir / 'feats.pkl'
    T = {}
    t0 = time.time()
    if reuse and fpath.exists():
        recs = pickle.load(open(fpath, 'rb')); tim = {}
        print(f'reusing {fpath} ({len(recs)} molecules)')
    else:
        hold = fixed_holdout(holdout) if holdout else select_holdout(E.L, n_np, n_e180, seed)
        print(f'holdout: {len(hold)} molecules ({sum(h["src_lib"] == NP_LIB for h in hold)} np-examples, '
              f'{sum(h["eligible_A"] for h in hold)} A-eligible)', flush=True)
        raw = load_query_rows([i for h in hold for i in h['q_rows']])
        T['load_queries'] = time.time() - t0
        recs, tim = compute_features(E, hold, raw)
        T['features'] = time.time() - t0 - T['load_queries']
        pickle.dump(recs, open(fpath, 'wb'))
    t1 = time.time()
    res, single, fold = evaluate(recs, cfg)
    T['ranker_cv'] = time.time() - t1
    table, text = summarize(recs, res, single)
    print(text, flush=True)
    json.dump(dict(table=table, timings=T, channel_timings=tim, n=len(recs)), open(outdir / 'summary.json', 'w'),
              indent=1, default=str)
    (outdir / 'summary.txt').write_text(text + '\n' + json.dumps(dict(timings=T, channel_timings=tim), default=str) + '\n')
    return table, text, T


def cv_ranker(cfg, tag='exp000', cols_zero=None):
    """Ranker fitted on ALL CV rows (A as M=0, B as M=1) -- for test submissions with a leak-free-trained ranker."""
    recs = pickle.load(open(config.OUTPUTS / 'cv' / tag / 'feats.pkl', 'rb'))
    parts = [p for p in (_stack(recs, 'A', cols_zero), _stack(recs, 'B', cols_zero)) if p is not None]
    X = np.vstack([p[0] for p in parts]); Y = np.concatenate([p[1] for p in parts]); M = np.concatenate([p[2] for p in parts])
    return Ranker(cfg).fit(X, Y, M)


__all__ = ['run_cv', 'select_holdout', 'evaluate', 'summarize', 'cv_ranker', 'FEATURE_NAMES']


In [ ]:
%%writefile /kaggle/working/casmi/cvx.py
"""Ranker-side experiments on CACHED CV features (outputs/cv/<tag>/feats.pkl) -- no channel recomputation.

  uv run python -m casmi.cvx cands --tag exp000             # once: recover candidate pool indices per window
  uv run python -m casmi.cvx run --exp exp002 --groups desc,desc_rel,formula [--ranker hgb|lgb_rank]

Same folds as cv.evaluate (rng(0).permutation % 5), same Ranker config, only 'full' variant. Reports per-split MRR,
np-weighted total (0.16*A + 0.45*B, C=0), fold std, and a paired bootstrap vs a baseline run (per-molecule rr is
saved to outputs/cv/<tag>/cvx_<exp>.npz).

LightGBM needs libomp: run with DYLD_LIBRARY_PATH=.venv/lib/python3.12/site-packages/torch/lib on macOS.
"""
from __future__ import annotations

import argparse
import pickle
import time

import numpy as np

from . import config
from .cv import NP_LIB, WEIGHTS, rr_from_scores

OUT = config.OUTPUTS / 'cv'


# --------------------------------------------------------------------------------------------- candidate recovery
def recover_candidates(tag='exp000'):
    from .candidates import Pool
    from .metric import score_keys_parallel
    cfg = config.CFG()
    recs = pickle.load(open(OUT / tag / 'feats.pkl', 'rb'))
    P = Pool(cfg)
    wins = []
    for r in recs:
        if r['B'] is None:
            wins.append(None); continue
        c = P.window(r['target'], cfg.PPM_WIN)
        if len(c) == 0:
            c = P.window(r['target'], cfg.PPM_FALLBACK)
        wins.append(c)
    allc = np.unique(np.concatenate([w for w in wins if w is not None]))
    keys = score_keys_parallel([P.smiles[i] for i in allc], workers=cfg.WORKERS)
    k_of = dict(zip(allc.tolist(), keys))
    out, bad = [], 0
    for r, w in zip(recs, wins):
        d = {}
        for s in 'ABC':
            p = r[s]
            if p is None:
                d[s] = None; continue
            if s == 'C':
                w_s = np.array([i for i in w if k_of[int(i)] != r['h']['mkey']], np.int64)
            else:
                w_s = w
            if len(w_s) == p['n'] and all(k_of[int(i)] == k for i, k in zip(w_s, p['keys'])):
                d[s] = w_s; continue
            # cap hit / order differs: match by key (first unused pool index with that key)
            bad += 1
            byk = {}
            for i in w_s:
                byk.setdefault(k_of[int(i)], []).append(int(i))
            d[s] = np.array([byk[k].pop(0) if byk.get(k) else -1 for k in p['keys']], np.int64)
        out.append(d)
    meta = dict(pidx=out, smiles=P.smiles, mass=P.mass, src=P.src)
    idx = np.unique(np.concatenate([v for d in out for v in d.values() if v is not None]))
    idx = idx[idx >= 0]
    small = dict(pidx=out, smiles={int(i): P.smiles[i] for i in idx}, mass={int(i): float(P.mass[i]) for i in idx},
                 src={int(i): int(P.src[i]) for i in idx})
    pickle.dump(small, open(OUT / tag / 'cands.pkl', 'wb'))
    print(f'recovered {len(out)} molecules, {bad} windows matched by key, {len(idx)} unique candidates')
    del meta
    return small


# --------------------------------------------------------------------------------------------- feature augmentation
def augment(recs, cands, groups):
    from .prior import describe_many, prior_features
    smi = [cands['smiles'][int(i)] for d in cands['pidx'] for v in d.values() if v is not None for i in v if i >= 0]
    D = describe_many(smi)
    new, names = [], None
    for r, d in zip(recs, cands['pidx']):
        rr = dict(r)
        for s in 'ABC':
            p = r[s]
            if p is None:
                continue
            pid = d[s]
            ok = pid >= 0
            sm = [cands['smiles'][int(i)] if i >= 0 else 'C' for i in pid]
            ms = np.array([cands['mass'][int(i)] if i >= 0 else r['target'] for i in pid])
            co = np.array([cands['src'][int(i)] == 0 if i >= 0 else 0 for i in pid], np.float32)
            Dx = dict(D); Dx.setdefault('C', (np.full(len(D[next(iter(D))][0]), np.nan, np.float32), None))
            ex, names = prior_features(sm, ms, r['target'], co, Dx, groups)
            ex[~ok] = 0.0
            rr[s] = dict(p, X=np.hstack([p['X'], ex]))
        new.append(rr)
    return new, names


def group_rel(recs, cols=(15, 5, 21, 0)):
    """Within-window relative features for key channels: gap to 2nd best, gap to best among OTHER candidates,
    softmax share. (window ~= same-formula group: 10 ppm)."""
    new = []
    for r in recs:
        rr = dict(r)
        for s in 'ABC':
            p = r[s]
            if p is None:
                continue
            X = p['X']; ex = []
            for c in cols:
                x = X[:, c].astype(np.float64)
                o = np.sort(x)[::-1]
                best, second = o[0], (o[1] if len(o) > 1 else o[0])
                other_best = np.where(x >= best, second, best)
                e = np.exp((x - best) * 3.0); sm = e / e.sum()
                ex += [x - other_best, np.full(len(x), best - second), sm]
            rr[s] = dict(p, X=np.hstack([X, np.column_stack(ex).astype(np.float32)]))
        new.append(rr)
    return new



def fp_block(recs, cands, tag='exp000'):
    """FPNet agreement split by fingerprint family (ECFP4 / ECFP6 / RDKit path / MACCS), plus confident-bit hinge
    counts and cosine(f, sigmoid z) over bits that VARY inside the window. Needs zlog.npy (cvx zlog)."""
    from .candidates import Pool
    from .features import _rank_norm, _z
    Z = np.load(OUT / tag / 'zlog.npy')
    bits = np.load(config.COCO_DIR / 'fp_bits.npy')
    blk = np.searchsorted([4096, 8192, 10240], bits, 'right')  # 0..3
    P = Pool(config.CFG())
    new = []
    chk = []
    for j, (r, d) in enumerate(zip(recs, cands['pidx'])):
        rr = dict(r)
        z = Z[j]
        for s in 'ABC':
            p = r[s]
            if p is None:
                continue
            pid = d[s]
            F = P.fps(np.maximum(pid, 0)).astype(np.float32)
            raw = F @ z
            if s == 'B' and len(raw) > 2:
                chk.append(np.corrcoef(_rank_norm(raw), p['X'][:, 16])[0, 1])
            ex = []
            for b in range(4):
                m = blk == b
                x = F[:, m] @ z[m]
                ex += [_z(x), _rank_norm(x), x - x.max()]
            var = (F.mean(0) > 0) & (F.mean(0) < 1)
            pr = 1 / (1 + np.exp(-z[var])); Fv = F[:, var]
            nv = max(int(var.sum()), 1)
            miss = ((1 - Fv) * (pr > 0.9)).sum(1); extra = (Fv * (pr < 0.1)).sum(1)
            cos = (Fv @ pr) / (np.linalg.norm(Fv, axis=1) * np.linalg.norm(pr) + 1e-9)
            ex += [miss, extra, _rank_norm(-(miss + extra)), _z(cos), _rank_norm(cos), np.full(len(raw), np.log(nv))]
            rr[s] = dict(p, X=np.hstack([p['X'], np.column_stack(ex).astype(np.float32)]))
        new.append(rr)
    print(f'fp_block: rank corr with cached fz_rank (split B) median={np.median(chk):.4f} min={np.min(chk):.4f}')
    return new



def frag2_feats(recs, cands, tag='exp000', workers=4):
    """MetFrag-lite v2 features (frag2.py): max over the molecule's spectra + window z/rank of key columns."""
    import multiprocessing as mp
    from .features import _rank_norm, _z
    from .frag2 import fragment_masses2, spec_scores
    from .spectra import _clean
    cfg = config.CFG()
    cp = OUT / tag / 'frag2_masses.pkl'
    FM = pickle.load(open(cp, 'rb')) if cp.exists() else {}
    need = sorted({cands['smiles'][int(i)] for d in cands['pidx'] for v in d.values() if v is not None for i in v} - set(FM))
    if need:
        with mp.get_context('spawn').Pool(workers) as p:
            FM.update(zip(need, p.map(fragment_masses2, need, chunksize=64)))
        pickle.dump(FM, open(cp, 'wb'))
    raw = pickle.load(open(OUT / tag / 'query_raw.pkl', 'rb'))
    new = []
    for r, d in zip(recs, cands['pidx']):
        rr = dict(r)
        specs = []
        for q in r['h']['q_rows']:
            sp = raw[q]
            m2, i2 = _clean(np.asarray(sp['mz'], np.float32), np.asarray(sp['it'], np.float32), cfg.INT_FLOOR,
                            cfg.MAX_PEAKS, 1.0, False)
            specs.append((np.asarray(m2, float), np.asarray(i2, float), sp['mode'], sp['adduct'], sp['prec']))
        cache = {}
        for s in 'ABC':
            p = r[s]
            if p is None:
                continue
            rows = []
            for i in d[s]:
                i = int(i)
                if i not in cache:
                    fm, fk = FM[cands['smiles'][i]]
                    v = np.stack([spec_scores(fm, fk, *sp) for sp in specs]) if specs else np.zeros((1, 8), np.float32)
                    cache[i] = np.concatenate([v.max(0), v.mean(0)[[1, 3]]])
                rows.append(cache[i])
            F = np.stack(rows).astype(np.float32)
            ex = [F]
            for c in (1, 3, 4):
                ex += [_z(F[:, c])[:, None], _rank_norm(F[:, c])[:, None], (F[:, c] - F[:, c].max())[:, None]]
            rr[s] = dict(p, X=np.hstack([p['X']] + ex).astype(np.float32))
        new.append(rr)
    return new


# --------------------------------------------------------------------------------------------- rankers
class LGBRank:
    """LightGBM LambdaRank grouped by (molecule, split); the class prior w1 enters as per-group weights."""

    def __init__(self, cfg, priors=None, seeds=(0, 1), params=None):
        self.priors = priors or cfg.W1_PRIORS; self.seeds = seeds
        self.params = dict(objective='lambdarank', learning_rate=0.03, num_leaves=31, min_data_in_leaf=50,
                           lambda_l2=1.0, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
                           lambdarank_truncation_level=25, eval_at=[25], verbose=-1, num_threads=0)
        if params:
            self.params.update(params)
        self.n_iter = self.params.pop('n_iter', 500)

    def fit(self, X, Y, M, G, sw=None):
        """sw: optional per-row weight multiplier, constant within a group (e.g. per molecule source library)."""
        import lightgbm as lgb
        o = np.argsort(G, kind='stable'); X, Y, M, G = X[o], Y[o], M[o], G[o]
        sw = np.ones(len(Y)) if sw is None else np.asarray(sw, float)[o]
        _, cnt = np.unique(G, return_counts=True)
        gm = M[np.cumsum(cnt) - 1]
        self.models = []
        for w1 in self.priors:
            gw = np.where(gm == 0, w1, 1 - w1)
            for sd in self.seeds:
                ds = lgb.Dataset(X, Y, group=cnt, weight=np.repeat(gw, cnt) * sw)
                self.models.append(lgb.train(dict(self.params, seed=sd), ds, self.n_iter))
        return self

    def predict(self, X):
        return np.mean([m.predict(X) for m in self.models], axis=0)


class Blend:
    """Mean of within-window rank scores of the HistGBM (binary) and LightGBM (lambdarank) rankers."""

    def __init__(self, a, b, wa=0.5):
        self.a, self.b, self.wa = a, b, wa

    def predict(self, X):
        from .features import _rank_norm
        return self.wa * (1 - _rank_norm(self.a.predict(X))) + (1 - self.wa) * (1 - _rank_norm(self.b.predict(X)))


def _stack(recs, split, gbase):
    X, Y, M, G = [], [], [], []
    for j, r in enumerate(recs):
        p = r[split]
        if p is None or p['n'] == 0:
            continue
        X.append(p['X']); Y.append(p['Y']); M.append(np.full(p['n'], 0 if split == 'A' else 1))
        G.append(np.full(p['n'], gbase + j))
    return X, Y, M, G


def evaluate(recs, cfg, ranker='hgb', n_folds=5, seed=0, drop_cols=None, lgb_params=None):
    from .ranker import Ranker
    rng = np.random.default_rng(seed)
    fold = rng.permutation(len(recs)) % n_folds
    rr = {s: np.full(len(recs), np.nan) for s in 'ABC'}
    keep = None
    for f in range(n_folds):
        tr = [recs[i] for i in range(len(recs)) if fold[i] != f]
        parts = [_stack(tr, 'A', 0), _stack(tr, 'B', 10 ** 6)]
        X = np.vstack([x for p in parts for x in p[0]]); Y = np.concatenate([x for p in parts for x in p[1]])
        M = np.concatenate([x for p in parts for x in p[2]]); G = np.concatenate([x for p in parts for x in p[3]])
        if drop_cols is not None:
            keep = np.setdiff1d(np.arange(X.shape[1]), drop_cols); X = X[:, keep]
        if ranker == 'hgb':
            rk = Ranker(cfg).fit(X, Y, M)
        elif ranker == 'blend':
            rk = Blend(Ranker(cfg).fit(X, Y, M), LGBRank(cfg, params=lgb_params).fit(X, Y, M, G))
        else:
            rk = LGBRank(cfg, params=lgb_params).fit(X, Y, M, G)
        for i in np.where(fold == f)[0]:
            for s in 'ABC':
                p = recs[i][s]
                if s == 'A' and not recs[i]['h']['eligible_A']:
                    continue
                if p is None:
                    rr[s][i] = 0.0; continue
                x = p['X'] if keep is None else p['X'][:, keep]
                rr[s][i] = rr_from_scores(p, rk.predict(x))
    return rr, fold


def report(recs, rr, fold, base=None, n_boot=2000):
    libs = np.array([r['h']['src_lib'] for r in recs])
    np_ = libs == NP_LIB
    A = np.nan_to_num(rr['A']); B = rr['B']
    permol = WEIGHTS['A'] * A + WEIGHTS['B'] * B  # np molecules are all A-eligible
    line = (f"A[np]={np.nanmean(rr['A'][np_]):.4f} B[np]={np.nanmean(B[np_]):.4f} "
            f"W[np]={permol[np_].mean():.4f} B[e180]={np.nanmean(B[~np_]):.4f}")
    fw = [permol[np_ & (fold == f)].mean() for f in range(fold.max() + 1)]
    line += f"  fold W[np] std={np.std(fw):.4f}"
    if base is not None:
        bA = np.nan_to_num(base['A']); bperm = WEIGHTS['A'] * bA + WEIGHTS['B'] * base['B']
        d = (permol - bperm)[np_]
        rng = np.random.default_rng(1)
        bs = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)])
        dB = (B - base['B'])[np_]
        line += (f"\n  delta W[np]={d.mean():+.4f} (boot se {bs.std():.4f}, P(>0)={np.mean(bs > 0):.2f}); "
                 f"dA[np]={np.nanmean((rr['A'] - base['A'])[np_]):+.4f} dB[np]={dB.mean():+.4f} "
                 f"dB[e180]={np.nanmean((B - base['B'])[~np_]):+.4f}; np mols up/down {int((d > 1e-9).sum())}/{int((d < -1e-9).sum())}")
    return line


def main(argv=None):
    ap = argparse.ArgumentParser('casmi.cvx')
    sub = ap.add_subparsers(dest='cmd', required=True)
    c = sub.add_parser('cands'); c.add_argument('--tag', default='exp000')
    z = sub.add_parser('zlog'); z.add_argument('--tag', default='exp000')
    k = sub.add_parser('cmp'); k.add_argument('--tag', default='exp000'); k.add_argument('--base', default='base'); k.add_argument('exps', nargs='+')
    r = sub.add_parser('run')
    r.add_argument('--tag', default='exp000'); r.add_argument('--exp', required=True)
    r.add_argument('--groups', default='', help='prior groups: desc,desc_rel,formula,coco')
    r.add_argument('--grouprel', action='store_true', help='add within-window gap/softmax features')
    r.add_argument('--fpblock', action='store_true', help='add per-family FPNet agreement features')
    r.add_argument('--frag2', action='store_true', help='add MetFrag-lite v2 features (frag2.py)')
    r.add_argument('--ranker', default='hgb', choices=['hgb', 'lgb_rank', 'blend'])
    r.add_argument('--lgb', default='', help='k=v,... LightGBM param overrides')
    r.add_argument('--base', default='base', help='exp name of baseline run for paired bootstrap')
    r.add_argument('--fold-seed', type=int, default=0)
    a = ap.parse_args(argv)
    if a.cmd == 'cands':
        recover_candidates(a.tag); return
    if a.cmd == 'zlog':
        cache_logits(a.tag); return
    if a.cmd == 'cmp':
        recs = pickle.load(open(OUT / a.tag / 'feats.pkl', 'rb'))
        base = dict(np.load(OUT / a.tag / f'cvx_{a.base}.npz'))
        for e in a.exps:
            rr = dict(np.load(OUT / a.tag / f'cvx_{e}.npz'))
            print(f'[{e}]  ' + report(recs, rr, rr['fold'], None if e == a.base else base))
        return
    cfg = config.CFG()
    t0 = time.time()
    recs = pickle.load(open(OUT / a.tag / 'feats.pkl', 'rb'))
    names = []
    if a.groups:
        cands = pickle.load(open(OUT / a.tag / 'cands.pkl', 'rb'))
        recs, names = augment(recs, cands, tuple(a.groups.split(',')))
    if a.grouprel:
        recs = group_rel(recs)
    if a.frag2:
        recs = frag2_feats(recs, pickle.load(open(OUT / a.tag / 'cands.pkl', 'rb')), a.tag)
    if a.fpblock:
        recs = fp_block(recs, pickle.load(open(OUT / a.tag / 'cands.pkl', 'rb')), a.tag)
    lp = {}
    for kv in filter(None, a.lgb.split(',')):
        k, v = kv.split('='); lp[k] = type(LGBRank(cfg).params.get(k, 1.0))(float(v)) if k != 'n_iter' else int(v)
    rr, fold = evaluate(recs, cfg, a.ranker, seed=a.fold_seed, lgb_params=lp)
    np.savez(OUT / a.tag / f'cvx_{a.exp}.npz', fold=fold, **rr)
    bp = OUT / a.tag / f'cvx_{a.base}.npz'
    base = dict(np.load(bp)) if bp.exists() and a.base != a.exp else None
    print(f'[{a.exp}] +{len(names)} prior cols grouprel={a.grouprel} fpblock={a.fpblock} frag2={a.frag2} ranker={a.ranker} {lp} ({time.time()-t0:.0f}s)')
    print('  ' + report(recs, rr, fold, base), flush=True)




def cache_logits(tag='exp000'):
    """FPNet z per CV molecule (same queries as feats.pkl) -> outputs/cv/<tag>/zlog.npy (NaN rows if no query)."""
    import glob
    from .cv import load_query_rows, make_query
    from .fpnet import ModelBank
    recs = pickle.load(open(OUT / tag / 'feats.pkl', 'rb'))
    bank = ModelBank(sorted(glob.glob(str(config.FP_MODEL_DIR / 'fp_*.pt'))))
    raw = load_query_rows([i for r in recs for i in r['h']['q_rows']])
    Z = np.full((len(recs), bank.nbits), np.nan, np.float32)
    for j, r in enumerate(recs):
        q = make_query(None, r['h'], raw)
        if q is not None:
            z = bank.logits(q.spectra)
            if z is not None:
                Z[j] = z
    np.save(OUT / tag / 'zlog.npy', Z)
    pickle.dump(raw, open(OUT / tag / 'query_raw.pkl', 'wb'))
    print('zlog', Z.shape, np.isnan(Z[:, 0]).sum(), 'missing')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/casmi/engine.py
"""Per-molecule scoring: 4 evidence channels -> 31 features for every candidate in the +-10 ppm window.

Channels (see research/04-pipeline-analysis.md):
  1. library   : entropy similarity vs library spectra in the same neutral-mass window   (class-1 evidence)
  2. analog    : mass-shifted entropy similarity vs one representative per structure within +-200 Da,
                 propagated to candidates as max_a sim(a)^p * Tanimoto(c, a)            (class-2 evidence)
  3. frag      : MetFrag-lite explained-intensity fraction
  4. fpnet     : spectrum->fingerprint logits z, candidate score f.z

`Mask` lets CV hide evidence per query (library spectra, analog representatives, pool members) without rebuilding
any index -- see cv.py.
"""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np

from . import config
from .features import N_FEAT, rank_features
from .metric import score_key
from .spectra import clean, search_clean, search_shift_clean


@dataclass
class Mask:
    """Evidence hidden from one query.
    excl_sids   : structures whose spectra are removed from the library side entirely (split B/C)
    src_sids/src_lib : structures whose spectra are removed only from library `src_lib` (split A)
    pool_excl_mkeys  : metric keys removed from the candidate pool (split C)"""
    excl_sids: np.ndarray = field(default_factory=lambda: np.zeros(0, np.int64))
    src_sids: np.ndarray = field(default_factory=lambda: np.zeros(0, np.int64))
    src_lib: str | None = None
    pool_excl_mkeys: frozenset = frozenset()

    def spectra_bad(self, L, idx):
        if len(idx) == 0:
            return np.zeros(0, bool)
        s = L.sid[idx]
        bad = np.isin(s, self.excl_sids) if len(self.excl_sids) else np.zeros(len(idx), bool)
        if len(self.src_sids) and self.src_lib is not None:
            bad |= np.isin(s, self.src_sids) & (L.lib[idx] == self.src_lib)
        return bad

    @property
    def empty(self):
        return not len(self.excl_sids) and not len(self.src_sids) and not self.pool_excl_mkeys


NO_MASK = Mask()


@dataclass
class Query:
    spectra: list          # dict(mz, it, prec, adduct, instrument, ce, mode)
    target: float          # neutral mass (median over spectra)

    @property
    def peaks(self):
        return [(s['mz'], s['it']) for s in self.spectra]

    @property
    def mode(self):
        return float(np.mean([s['mode'] for s in self.spectra]))


class Engine:
    def __init__(self, cfg: config.CFG, L, pool, bank=None, frag=None, prefer_instrument='timsTOF'):
        self.cfg, self.L, self.pool, self.bank, self.frag = cfg, L, pool, bank, frag
        L.build_rep(prefer_instrument if cfg.ANALOG_MATCH_INSTR else None)
        self.key2sid = {k: i for i, k in enumerate(L.s_key)}
        self.sid2pidx = np.array([pool.k2i.get(k, -1) for k in L.s_key], np.int64)
        self.pidx2sid = np.full(len(pool.keys), -1, np.int64)
        ok = self.sid2pidx >= 0
        self.pidx2sid[self.sid2pidx[ok]] = np.where(ok)[0]

    # ------------------------------------------------------------------ channel 1
    def lib_sim(self, q: Query, mask: Mask = NO_MASK):
        """{sid: max entropy similarity} over library spectra within +-PPM_WIN of the target."""
        cfg, L = self.cfg, self.L
        cand = L.window(q.target, q.target * cfg.PPM_WIN / 1e6)
        if len(cand) and not mask.empty:
            cand = cand[~mask.spectra_bad(L, cand)]
        if len(cand) == 0:
            return {}
        best = np.full(len(cand), -1.0, np.float32)
        for mz, it in q.peaks:
            qm, qp = clean(mz, it, cfg)
            if len(qm) == 0:
                continue
            best = np.maximum(best, search_clean(qm, qp, cand, L.off, L.mz, L.it, cfg.MZ_TOL))
        agg = {}
        for s, v in zip(L.sid[cand].tolist(), best.tolist()):
            if v >= 0 and v > agg.get(s, -1.0):
                agg[s] = v
        return agg

    # ------------------------------------------------------------------ channel 2
    def analog_sim(self, q: Query, mask: Mask = NO_MASK, rep_override=None):
        """[(sid, sim)] top N_ANALOG by mass-shifted entropy similarity over +-ANALOG_WIN Da."""
        cfg, L = self.cfg, self.L
        lo = np.searchsorted(L.rep_nm, q.target - cfg.ANALOG_WIN, 'left')
        hi = np.searchsorted(L.rep_nm, q.target + cfg.ANALOG_WIN, 'right')
        cand = L.rep[lo:hi].copy(); csid = L.rep_sid[lo:hi]; cnm = L.rep_nm[lo:hi].copy()
        if len(cand) == 0:
            return []
        keep = np.ones(len(cand), bool)
        if len(mask.excl_sids):
            keep &= ~np.isin(csid, mask.excl_sids)
        if rep_override:
            pos = {s: i for i, s in enumerate(csid.tolist()) if s in rep_override}
            for s, r in rep_override.items():
                if s in pos:
                    if r < 0:
                        keep[pos[s]] = False
                    else:
                        cand[pos[s]] = r; cnm[pos[s]] = L.nm[r]
        cand, csid, cnm = cand[keep], csid[keep], cnm[keep]
        if len(cand) == 0:
            return []
        shift = (q.target - cnm).astype(np.float32)
        best = np.full(len(cand), -1.0, np.float32)
        any_q = False
        for mz, it in q.peaks:
            qm, qp = clean(mz, it, cfg)
            if len(qm) == 0:
                continue
            any_q = True
            best = np.maximum(best, search_shift_clean(qm, qp, cand, L.off, L.mz, L.it, cfg.MZ_TOL, shift))
        if not any_q:
            return []
        # one rep per sid -> agg == best; library spectra with no peaks keep 0 (as in the notebook)
        best = np.maximum(best, 0.0)
        o = np.argsort(-best, kind='stable')[:cfg.N_ANALOG]
        return list(zip(csid[o].tolist(), best[o].tolist()))

    # ------------------------------------------------------------------ channel 4
    def logits(self, q: Query):
        if self.bank is None or not self.cfg.USE_FP_MODEL:
            return None
        return self.bank.logits(q.spectra)

    # ------------------------------------------------------------------ candidates + features
    def candidates(self, q: Query, lib_hits, analogs, zlog, pool_excl_mkeys=frozenset()):
        cfg, pool = self.cfg, self.pool
        cand = pool.window(q.target, cfg.PPM_WIN)
        if len(cand) == 0:
            cand = pool.window(q.target, cfg.PPM_FALLBACK)
        if len(cand) and pool_excl_mkeys:
            cand = cand[np.array([score_key(pool.smiles[c]) not in pool_excl_mkeys for c in cand], bool)]
        if len(cand) == 0:
            return None
        lv = np.array([lib_hits.get(self.pidx2sid[c], 0.0) if self.pidx2sid[c] >= 0 else 0.0 for c in cand], np.float32)
        if cfg.CAND_CAP and len(cand) > cfg.CAND_CAP:
            coarse = lv * 100.0
            if zlog is not None:
                mz = pool.fps(cand).astype(np.float32) @ zlog
                coarse = coarse + (mz - mz.mean()) / max(float(mz.std()), 1e-9)
            else:
                coarse = coarse - np.abs(pool.mass[cand] - q.target)
            keep = np.argsort(-coarse)[:cfg.CAND_CAP]
            cand, lv = cand[keep], lv[keep]
        cfp = pool.fps(cand)
        csmi = [pool.smiles[c] for c in cand]
        ids, sims = [], []
        for s, v in analogs:
            i = self.sid2pidx[s]
            if i >= 0:
                ids.append(i); sims.append(v)
        afp = pool.fps(np.array(ids)) if ids else None
        fsc = self.frag.scores(csmi, q.peaks, q.mode, cfg) if (self.frag is not None and cfg.USE_FRAG) else None
        X = rank_features(cfp, lv, afp, np.array(sims, np.float32), zlog, fsc, p_sim=cfg.SIM_POWER)[:, :N_FEAT]
        return dict(pidx=cand, smiles=csmi, X=X, lv=lv, top_analog=float(sims[0]) if sims else 0.0)


In [ ]:
%%writefile /kaggle/working/casmi/features.py
"""31 ranker features per candidate (verbatim port of the notebook's rank_features).

Deliberately EXCLUDES pool provenance (src / np_likeness): in the class-2 simulation the answer is always a
training-library structure, so those columns leak.
"""
from __future__ import annotations

import numpy as np

N_FEAT = 31
FEATURE_NAMES = [
    'lv', 'lv_rank', 'lvmax', 'lv_minus_max', 'lv_pos',
    'ap', 'ap_rank', 'apmax', 'ap_minus_max',
    'a1', 'best_tan', 'top_tan', 'mean_tan', 'top_sim', 'log_nc',
    'fz_z', 'fz_rank', 'fz_minus_max', 'fzn_z', 'fzn_rank', 'fz_is_max',
    'fr', 'fr_rank', 'fr_minus_max', 'fr_z',
    'lv_x_mr', 'ap_x_mr', 'agree_lib', 'agree_analog', 'agree_x_lvmax', 'corr_lv_model',
]


def _rank_norm(x):
    o = np.argsort(-x); r = np.empty(len(x)); r[o] = np.arange(len(x)); return r / max(1, len(x) - 1)


def _z(x):
    s = x.std()
    return (x - x.mean()) / s if s > 1e-9 else np.zeros_like(x)


def rank_features(cand_fp, cand_lib, analog_fp, analog_sim, model_logits=None, frag=None, p_sim=3.0):
    nc = cand_fp.shape[0]
    cf = cand_fp.astype(np.float32); cs = cf.sum(1)
    lv = np.asarray(cand_lib, np.float32)
    lvmax = float(lv.max()) if nc else 0.0
    if analog_fp is not None and len(analog_sim):
        af = analog_fp.astype(np.float32); asum = af.sum(1)
        inter = cf @ af.T
        tan = inter / (cs[:, None] + asum[None, :] - inter + 1e-9)
        w = np.clip(np.asarray(analog_sim, np.float32), 0, None)
        ap = (tan * (w ** p_sim)[None, :]).max(1)
        a1 = (tan * w[None, :]).max(1)
        best_tan = tan.max(1); top_tan = tan[:, 0]; top_sim = float(w[0])
        mean_tan = (tan * (w ** p_sim)[None, :]).sum(1) / ((w ** p_sim).sum() + 1e-9)
    else:
        ap = a1 = best_tan = top_tan = mean_tan = np.zeros(nc, np.float32); top_sim = 0.0
    apmax = float(ap.max()) if nc else 0.0
    if model_logits is not None:
        raw = cf @ np.asarray(model_logits, np.float32)
        nrm = raw / np.sqrt(np.maximum(cs, 1.0))
        mfeat = [_z(raw), _rank_norm(raw), raw - raw.max(), _z(nrm), _rank_norm(nrm),
                 (raw == raw.max()).astype(np.float32)]
    else:
        mfeat = [np.zeros(nc, np.float32)] * 6
    if model_logits is not None and nc:
        mr = _rank_norm(cf @ np.asarray(model_logits, np.float32))
        lbest = int(np.argmax(lv)) if lvmax > 0 else -1
        agree = float(1.0 - mr[lbest]) if lbest >= 0 else 0.0
        abest = int(np.argmax(ap)) if apmax > 0 else -1
        agree_a = float(1.0 - mr[abest]) if abest >= 0 else 0.0
        xfeat = [lv * (1.0 - mr), ap * (1.0 - mr), np.full(nc, agree), np.full(nc, agree_a),
                 np.full(nc, agree * lvmax), np.full(nc, float(np.corrcoef(lv, -mr)[0, 1]) if lv.std() > 1e-9 else 0.0)]
    else:
        xfeat = [np.zeros(nc, np.float32)] * 6
    if frag is not None:
        fr = np.asarray(frag, np.float32)
        ffeat = [fr, _rank_norm(fr), fr - fr.max() if nc else fr, _z(fr)]
    else:
        ffeat = [np.zeros(nc, np.float32)] * 4
    return np.column_stack([
        lv, _rank_norm(lv), np.full(nc, lvmax), lv - lvmax, (lv > 0).astype(float),
        ap, _rank_norm(ap), np.full(nc, apmax), ap - apmax,
        a1, best_tan, top_tan, mean_tan, np.full(nc, top_sim),
        np.full(nc, np.log(max(nc, 1))),
        *mfeat, *ffeat, *xfeat,
    ]).astype(np.float32)


In [ ]:
%%writefile /kaggle/working/casmi/fpnet.py
"""Spectrum -> 6,930-bit fingerprint Transformer (CSI:FingerID-style), ranked by f.z. Verbatim port of the notebook."""
from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

MAX_PEAKS = 128
ADDUCT_LIST = ["[M+H]+", "[M+NH4]+", "[M+Na]+", "[M+K]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M]+",
               "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+C2H4O2-H]-", "[M+Cl]-", "[M]-",
               "[M+2H]2+", "[M-2H]-", "[2M+H]+", "[2M+Na]+", "[2M+NH4]+", "[2M-H]-", "[2M+K]+",
               "[2M+CH2O2-H]-", "[2M+C2H4O2-H]-", "[2M+Na-2H]-", "[M+Na-2H]-", "[M-H2O]+", "<unk>"]
ADDUCT_IX = {a: i for i, a in enumerate(ADDUCT_LIST)}
INSTR_LIST = ["timsTOF", "Orbitrap", "QTOF", "IT", "other"]


def instr_family(s):
    if s is None:
        return 4
    t = str(s).lower()
    if 'timstof' in t: return 0
    if 'orbitrap' in t or 'qft' in t or 'ftms' in t or 'hybrid ft' in t or 'itft' in t or 'exactive' in t: return 1
    if 'tof' in t: return 2
    if 'trap' in t or 'qq' in t: return 3
    return 4


def prep_peaks(mz, inten, prec_mz, max_peaks=MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):
    """Filter -> window-diversified top-N -> sort by m/z. Returns (mz, sqrt-intensity)."""
    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)
    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)
    keep = (mz <= prec_mz + 1.5)
    mz, it = mz[keep], it[keep]
    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)
    mx = it.max()
    if mx <= 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)
    keep = it >= floor * mx
    mz, it = mz[keep], it[keep]
    if len(mz) > max_peaks:
        order = np.argsort(-it)
        bucket = (mz // win).astype(np.int64)
        cnt = {}; sel = []
        for i in order:
            b = bucket[i]; c = cnt.get(b, 0)
            if c < per_win: cnt[b] = c + 1; sel.append(i)
        sel = np.array(sel)
        if len(sel) > max_peaks:
            sel = sel[np.argsort(-it[sel])[:max_peaks]]
        elif len(sel) < max_peaks:
            ss = set(sel.tolist())
            rest = np.array([i for i in order if i not in ss])
            need = max_peaks - len(sel)
            if len(rest): sel = np.concatenate([sel, rest[:need]])
        mz, it = mz[sel], it[sel]
    o = np.argsort(mz)
    mz, it = mz[o], it[o]
    v = np.sqrt(it / it.max())
    return mz.astype(np.float32), v.astype(np.float32)


class SinEmb(nn.Module):
    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):
        super().__init__()
        n = dim // 2
        wav = torch.pow(10.0, (hi - lo) * torch.pow(torch.linspace(0, 1, n), power) + lo)
        self.register_buffer('inv', (2 * math.pi) / wav)

    def forward(self, x):
        a = x.unsqueeze(-1) * self.inv
        return torch.cat([torch.sin(a), torch.cos(a)], -1)


class Block(nn.Module):
    def __init__(self, d, h, drop):
        super().__init__(); self.h = h
        self.n1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)
        self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(drop)

    def forward(self, x, pad):
        B, N, D = x.shape; y = self.n1(x)
        q, k, v = self.qkv(y).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        m = (~pad)[:, None, None, :]
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, N, D)))
        return x + self.drop(self.ff(self.n2(x)))


class FPNet(nn.Module):
    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):
        super().__init__()
        self.d = d
        self.mz_emb = SinEmb(d); self.nl_emb = SinEmb(d)
        self.pk = nn.Linear(2 * d + 1, d)
        self.prec_emb = SinEmb(d)
        self.ad = nn.Embedding(len(ADDUCT_LIST), d)
        self.ins = nn.Embedding(len(INSTR_LIST), d)
        self.gl = nn.Linear(d + 3, d)
        self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))

    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):
        B, N = mz.shape
        nl = (prec[:, None] - mz).clamp(min=0)
        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))
        g = self.gl(torch.cat([self.prec_emb(prec), (ce / 100.0).unsqueeze(-1), mode.unsqueeze(-1),
                               torch.log1p(prec).unsqueeze(-1) / 10.0], -1)) + self.ad(ad) + self.ins(ins)
        x = torch.cat([g.unsqueeze(1), p], 1)
        pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=pad.device), pad], 1)
        for b in self.blocks: x = b(x, pad)
        x = self.norm(x)
        cls = x[:, 0]
        msk = (~pad[:, 1:]).float().unsqueeze(-1)
        mean = (x[:, 1:] * msk).sum(1) / msk.sum(1).clamp(min=1)
        return self.head(torch.cat([cls, mean], -1))


class ModelBank:
    """fp_merged_* models get one merged peak list; fp_single_* get each spectrum. Final z = mean of the two views."""

    def __init__(self, paths, device='cpu'):
        self.dev = device
        self.single, self.merged = [], []
        for pth in sorted(paths):
            ck = torch.load(pth, map_location='cpu', weights_only=False)
            net = FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).to(device).eval()
            net.load_state_dict(ck['model'])
            (self.merged if 'merged' in str(pth).split('/')[-1] else self.single).append(net)
            self.nbits = ck['nbits']
            print(f'  loaded {str(pth).split("/")[-1]}: d={ck["d"]} layers={ck["layers"]} step={ck.get("step")}')
        torch.set_num_threads(max(1, torch.get_num_threads()))

    @torch.no_grad()
    def logits(self, spectra):
        """spectra: list of dict(mz, it, prec, adduct, instrument, ce(list|float|None), mode(+1/-1)).
        Exactly the notebook's model_logits(sub)."""
        out = []
        if self.single:
            za = self._logits_from(spectra, self.single)
            if za is not None: out.append(za)
        if self.merged:
            mz, it = merge_peaks(spectra)
            r0 = spectra[0]
            zb = self._logits_raw([(mz, it)], self.merged, float(np.median([s['prec'] for s in spectra])),
                                  r0['adduct'], r0['instrument'], 25.0, float(np.mean([s['mode'] for s in spectra])))
            if zb is not None: out.append(zb)
        return np.mean(out, axis=0).astype(np.float32) if out else None

    def _run(self, nets, mz, it, pad, prec, ad, ins, ce, mode):
        T = lambda x: torch.as_tensor(x, device=self.dev)
        args = (T(mz), T(it), T(pad), T(prec), T(ad), T(ins), T(ce), T(mode))
        return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)

    def _logits_from(self, spectra, nets):
        P, rows = [], []
        for r in spectra:
            a, b = prep_peaks(r['mz'], r['it'], float(r['prec']))
            if len(a): P.append((a, b)); rows.append(r)
        if not P: return None
        B = len(P); N = max(len(a) for a, _ in P)
        mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)
        for i, (a, b) in enumerate(P):
            mz[i, :len(a)] = a; it[i, :len(b)] = b; pad[i, :len(a)] = False

        def ce_of(r):
            v = r.get('ce')
            try:
                return float(np.mean(np.atleast_1d(v))) if v is not None and len(np.atleast_1d(v)) else 25.0
            except Exception:
                return 25.0
        return self._run(nets, mz, it, pad,
                         np.array([float(r['prec']) for r in rows], np.float32),
                         np.array([ADDUCT_IX.get(r['adduct'], ADDUCT_IX['<unk>']) for r in rows]),
                         np.array([instr_family(r['instrument']) for r in rows]),
                         np.array([ce_of(r) for r in rows], np.float32),
                         np.array([float(r['mode']) for r in rows], np.float32))

    def _logits_raw(self, pairs, nets, prec, adduct, instrument, ce, mode):
        P = [prep_peaks(mz, it, prec) for mz, it in pairs]
        P = [(a, b) for a, b in P if len(a)]
        if not P: return None
        B = len(P); N = max(len(a) for a, _ in P)
        mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)
        for i, (a, b) in enumerate(P):
            mz[i, :len(a)] = a; it[i, :len(b)] = b; pad[i, :len(a)] = False
        return self._run(nets, mz, it, pad, np.full(B, prec, np.float32),
                         np.full(B, ADDUCT_IX.get(adduct, ADDUCT_IX['<unk>'])), np.full(B, instr_family(instrument)),
                         np.full(B, ce, np.float32), np.full(B, mode, np.float32))


def merge_peaks(spectra):
    """All of a molecule's peaks collapsed into one pseudo-spectrum (near-duplicate m/z merged, stronger kept)."""
    mz = np.concatenate([np.asarray(s['mz'], float) for s in spectra])
    it = np.concatenate([np.asarray(s['it'], float) / max(float(np.asarray(s['it'], float).max()), 1e-9)
                         if len(s['it']) else np.zeros(0) for s in spectra])
    o = np.argsort(mz); mz, it = mz[o], it[o]
    keep = np.ones(len(mz), bool)
    for j in range(1, len(mz)):
        if mz[j] - mz[j - 1] < 0.005:
            if it[j] >= it[j - 1]: keep[j - 1] = False
            else: keep[j] = False
    return mz[keep], it[keep]


In [ ]:
%%writefile /kaggle/working/casmi/frag.py
"""MetFrag-lite: fraction of sqrt-intensity explained by 1- or 2-bond-break fragments (+-2 H). Verbatim port;
fragment masses are cached per SMILES and computed with a persistent worker pool."""
from __future__ import annotations

import multiprocessing as mp

import numpy as np

from .spectra import _clean

AMU = {'C': 12.0, 'H': 1.00782503207, 'N': 14.0030740048, 'O': 15.9949146196, 'P': 30.97376163,
       'S': 31.97207100, 'F': 18.99840322, 'Cl': 34.96885268, 'Br': 78.9183371, 'I': 126.904473,
       'Na': 22.9897692809, 'K': 38.96370668, 'Si': 27.9769265325, 'B': 11.0093054, 'Se': 79.9165213}
H = AMU['H']; PROTON = H - 0.00054857990


def mol_graph(smi):
    from rdkit import Chem, RDLogger
    RDLogger.DisableLog('rdApp.*')
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    n = m.GetNumAtoms()
    w = np.zeros(n)
    for a in m.GetAtoms():
        w[a.GetIdx()] = AMU.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H
    if (w == 0).any(): return None
    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()]
    return w, bonds, n


def _components(n, bonds, drop):
    adj = [[] for _ in range(n)]
    for i, (a, b) in enumerate(bonds):
        if i in drop: continue
        adj[a].append(b); adj[b].append(a)
    seen = np.zeros(n, bool); comps = []
    for s in range(n):
        if seen[s]: continue
        stack = [s]; seen[s] = True; cur = [s]
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if not seen[v]: seen[v] = True; stack.append(v); cur.append(v)
        comps.append(cur)
    return comps


def fragment_masses(smi, max_breaks=2, max_bonds=34):
    g = mol_graph(smi)
    if g is None: return np.zeros(0)
    w, bonds, n = g
    nb = len(bonds)
    if nb == 0 or nb > max_bonds: return np.array([w.sum()])
    out = {w.sum()}
    for i in range(nb):
        for c in _components(n, bonds, {i}):
            out.add(float(w[c].sum()))
    if max_breaks >= 2:
        for i in range(nb):
            for j in range(i + 1, nb):
                for c in _components(n, bonds, {i, j}):
                    out.add(float(w[c].sum()))
    return np.array(sorted(out))


def explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):
    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0
    ion = []
    for dh in h_shifts:
        ion.append(frag_mass + dh * H + (PROTON if mode > 0 else -PROTON))
    ion = np.sort(np.concatenate(ion))
    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()
    if tot <= 0: return 0.0
    idx = np.searchsorted(ion, peak_mz)
    ok = np.zeros(len(peak_mz), bool)
    for off in (-1, 0):
        k = np.clip(idx + off, 0, len(ion) - 1)
        ok |= np.abs(ion[k] - peak_mz) <= tol
    return float(w[ok].sum() / tot)


def _frag_masses(smi):
    try:
        return fragment_masses(smi)
    except Exception:
        return np.zeros(0)


class Fragmenter:
    def __init__(self, workers=4):
        self.cache = {}
        self.workers = workers
        self._pool = None

    def _p(self):
        if self._pool is None and self.workers > 1:
            self._pool = mp.get_context('spawn').Pool(self.workers)
        return self._pool

    def masses(self, smiles):
        need = [s for s in dict.fromkeys(smiles) if s not in self.cache]
        if need:
            p = self._p()
            res = p.map(_frag_masses, need, chunksize=4) if p is not None and len(need) > 8 else list(map(_frag_masses, need))
            self.cache.update(zip(need, res))
        return [self.cache[s] for s in smiles]

    def scores(self, cand_smiles, specs, mode, cfg):
        """Max over the molecule's spectra of the explain-score, per candidate."""
        frags = self.masses(list(cand_smiles))
        peaks = []
        for mz, it in specs:
            m2, i2 = _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32), cfg.INT_FLOOR, cfg.MAX_PEAKS, 1.0, False)
            peaks.append((np.asarray(m2, float), np.asarray(i2, float)))
        out = np.zeros(len(cand_smiles), np.float32)
        for j, f in enumerate(frags):
            out[j] = max((explain_score(f, a, b, mode=mode, tol=cfg.MZ_TOL) for a, b in peaks), default=0.0)
        return out

    def close(self):
        if self._pool is not None:
            self._pool.close(); self._pool = None


In [ ]:
%%writefile /kaggle/working/casmi/frag2.py
"""MetFrag-lite v2 (CV experiment): fragment masses tagged by number of broken bonds, larger bond cap (the v1 port
returns only the precursor mass for molecules with > 34 bonds, i.e. most glycosides), adduct-aware ion types, and a
chance-match correction. Features are added on top of the 31 (see cvx.py --frag2).
"""
from __future__ import annotations

import numpy as np

from .frag import H, PROTON, _components, mol_graph

NA, NH4 = 22.9897692809 - 0.00054857990, 14.0030740048 + 4 * 1.00782503207 - 0.00054857990
H2O = 2 * 1.00782503207 + 15.9949146196


def fragment_masses2(smi, max_bonds=60):
    """(masses, n_breaks) unique by rounded mass keeping the fewest breaks."""
    try:
        g = mol_graph(smi)
    except Exception:
        g = None
    if g is None:
        return np.zeros(0), np.zeros(0, np.int8)
    w, bonds, n = g
    nb = len(bonds)
    best = {round(float(w.sum()), 4): 0}

    def add(m, k):
        r = round(float(m), 4)
        if best.get(r, 9) > k:
            best[r] = k

    if nb:
        for i in range(nb):
            for c in _components(n, bonds, {i}):
                add(w[c].sum(), 1)
        if nb <= max_bonds:
            for i in range(nb):
                for j in range(i + 1, nb):
                    for c in _components(n, bonds, {i, j}):
                        add(w[c].sum(), 2)
    ks = sorted(best)
    return np.array(ks), np.array([best[k] for k in ks], np.int8)


def _match(ion, mz, tol):
    if len(ion) == 0 or len(mz) == 0:
        return np.zeros(len(mz), bool)
    ion = np.sort(ion)
    idx = np.searchsorted(ion, mz)
    ok = np.zeros(len(mz), bool)
    for off in (-1, 0):
        k = np.clip(idx + off, 0, len(ion) - 1)
        ok |= np.abs(ion[k] - mz) <= tol
    return ok


def spec_scores(fm, fk, mz, it, mode, adduct, prec):
    """One spectrum -> vector of explanation scores for one candidate."""
    keep = mz < prec - 0.5  # fragment peaks only (precursor explained by every isomer)
    mz, it = mz[keep], it[keep]
    if len(mz) == 0 or len(fm) == 0:
        return np.zeros(8, np.float32)
    w = np.sqrt(it); tot = w.sum() + 1e-12
    ch = PROTON if mode > 0 else -PROTON
    hs = np.arange(-2, 3) * H

    def ions(m, extra=()):
        base = [m + h + ch for h in hs]
        for e in extra:
            base += [m + h + e for h in hs]
        return np.concatenate(base) if len(m) else np.zeros(0)

    extra = []
    if isinstance(adduct, str) and 'Na' in adduct:
        extra.append(NA)
    if isinstance(adduct, str) and 'NH4' in adduct:
        extra.append(NH4)
    m1 = fm[fk <= 1]
    ok1 = _match(ions(m1), mz, 0.01)
    ok2 = _match(ions(fm), mz, 0.01)
    ok2t = _match(ions(fm), mz, 0.005)
    ok2a = ok2 | _match(ions(fm, extra), mz, 0.01) | _match(ions(fm - H2O), mz, 0.01)
    # chance level: fraction of the m/z axis covered by ion windows
    span = max(float(prec), 50.0)
    chance = min(1.0, len(fm) * len(hs) * 2 * 0.01 / span)
    f2 = ok2 @ w / tot
    top = np.argsort(-it)[:10]
    return np.array([ok1 @ w / tot, f2, ok2t @ w / tot, ok2a @ w / tot, f2 - chance, chance,
                     ok2[top].mean(), ok2 @ it / (it.sum() + 1e-12)], np.float32)


FRAG2_NAMES = ['fr1', 'fr2', 'fr2_tight', 'fr2_adduct', 'fr2_minus_chance', 'fr_chance', 'fr2_top10', 'fr2_lin']


In [ ]:
%%writefile /kaggle/working/casmi/library.py
"""Reference spectral library (train.parquet) as a compact on-disk cache + mass index.

Cache (data/cache/lib_*):
  lib_off.npy / lib_mz.npy / lib_it.npy   pre-cleaned peaks (CSR, float32) -- see spectra.clean_batch
  lib_meta.parquet                        per spectrum: sid, lib, instrument, adduct, mode, precursor_mz, nm, npk_raw, ce
  structs.parquet                         per structure (unique inchikey14): sid, inchikey14, smiles, formula, mkey
mkey = metric score key (tautomer-canonical InChIKey14) -- used for CV masking/labels, never as a feature.
"""
from __future__ import annotations

import time

import numpy as np
import polars as pl
import pyarrow.parquet as pq

from . import chem, config
from .spectra import clean_batch

C = config.CACHE


def _lib_paths():
    return dict(off=C / 'lib_off.npy', mz=C / 'lib_mz.npy', it=C / 'lib_it.npy',
                meta=C / 'lib_meta.parquet', structs=C / 'structs.parquet')


def build_library_cache(cfg: config.CFG, force=False):
    P = _lib_paths()
    if all(p.exists() for p in P.values()) and not force:
        print('library cache exists'); return
    C.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    # ---- structures
    lf = pl.scan_parquet(config.TRAIN)
    st = (lf.select('inchikey14', 'normalized_smiles', 'molecular_formula').with_row_index('row')
            .group_by('inchikey14').agg(pl.col('row').min(), pl.col('normalized_smiles').first(),
                                        pl.col('molecular_formula').first())
            .sort('row').drop('row').with_row_index('sid').collect())
    print(f'structures: {len(st):,}  ({time.time()-t0:.0f}s)', flush=True)
    from .metric import score_keys_parallel
    mk = score_keys_parallel(st['normalized_smiles'].to_list(), workers=cfg.WORKERS)
    st = st.with_columns(pl.Series('mkey', mk, dtype=pl.String))
    st.write_parquet(P['structs'])
    print(f'metric keys done: {st["mkey"].null_count()} null, {st["mkey"].n_unique():,} unique '
          f'({time.time()-t0:.0f}s)', flush=True)
    k2sid = dict(zip(st['inchikey14'].to_list(), st['sid'].to_list()))
    # ---- spectra, row group by row group
    pf = pq.ParquetFile(config.TRAIN)
    metas, offs, mzs, its = [], [], [], []
    base = 0
    fm_cache = {}
    for rg in range(pf.num_row_groups):
        t = pf.read_row_group(rg, columns=['ingest_lib', 'inchikey14', 'molecular_formula', 'instrument_type', 'adduct',
                                           'ionization_mode', 'precursor_mz', 'ms2_mzs', 'ms2_normalized_intensities',
                                           'collision_energy_ev'])
        mzc = t.column('ms2_mzs').combine_chunks(); itc = t.column('ms2_normalized_intensities').combine_chunks()
        off = mzc.offsets.to_numpy().astype(np.int64)
        off = off - off[0]
        mz = mzc.values.to_numpy(zero_copy_only=False).astype(np.float32)
        it = itc.values.to_numpy(zero_copy_only=False).astype(np.float32)
        noff, cmz, cit = clean_batch(off, mz, it, cfg.INT_FLOOR, cfg.MAX_PEAKS, cfg.INT_POWER, cfg.ENT_WEIGHT)
        offs.append(noff[1:] + base); base += int(noff[-1]); mzs.append(cmz); its.append(cit)
        df = pl.from_arrow(t.drop(['ms2_mzs', 'ms2_normalized_intensities']))
        fo = df['molecular_formula'].to_list()
        for f in set(fo):
            if f not in fm_cache:
                fm_cache[f] = chem.formula_mass(f)
        nmf = np.array([fm_cache[f] for f in fo], np.float64)
        nmp = chem.neutral_mass(df['precursor_mz'].to_numpy(), df['adduct'].to_numpy())
        ce = df['collision_energy_ev'].list.mean().fill_null(-1.0)
        metas.append(pl.DataFrame(dict(
            sid=np.array([k2sid[k] for k in df['inchikey14'].to_list()], np.int32),
            lib=df['ingest_lib'], instrument=df['instrument_type'], adduct=df['adduct'], mode=df['ionization_mode'],
            precursor_mz=df['precursor_mz'], nm_formula=nmf, nm_prec=nmp, npk_raw=np.diff(off).astype(np.int32),
            ce=ce)))
        print(f'  row group {rg+1}/{pf.num_row_groups}: {len(off)-1:,} spectra, {len(cmz):,} cleaned peaks '
              f'({time.time()-t0:.0f}s)', flush=True)
    np.save(P['off'], np.concatenate([np.zeros(1, np.int64)] + offs))
    np.save(P['mz'], np.concatenate(mzs)); np.save(P['it'], np.concatenate(its))
    pl.concat(metas).write_parquet(P['meta'])
    print(f'library cache written ({time.time()-t0:.0f}s)', flush=True)


class Library:
    def __init__(self, cfg: config.CFG):
        P = _lib_paths()
        t0 = time.time()
        self.cfg = cfg
        self.off = np.load(P['off']); self.mz = np.load(P['mz']); self.it = np.load(P['it'])
        meta = pl.read_parquet(P['meta'])
        self.meta = meta
        self.sid = meta['sid'].to_numpy()
        self.lib = meta['lib'].to_numpy()
        self.instrument = meta['instrument'].to_numpy()
        nmf = meta['nm_formula'].to_numpy(); nmp = meta['nm_prec'].to_numpy()
        self.nm = np.where(np.isfinite(nmf), nmf, nmp) if cfg.LIB_MASS_FORMULA else nmp
        ok = np.isfinite(self.nm)
        self.order = np.argsort(np.where(ok, self.nm, 1e18), kind='mergesort')
        self.n_ok = int(ok.sum())
        self.snm = self.nm[self.order][:self.n_ok]
        self.npk = meta['npk_raw'].to_numpy()
        st = pl.read_parquet(P['structs'])
        self.structs = st
        self.s_key = st['inchikey14'].to_numpy()
        self.s_smiles = st['normalized_smiles'].to_numpy()
        self.s_mkey = st['mkey'].to_numpy()
        # spectra grouped by structure (for per-query masking / representative re-selection)
        o = np.argsort(self.sid, kind='mergesort')
        self._by_sid = o
        self._sid_bounds = np.searchsorted(self.sid[o], np.arange(len(st) + 1))
        # mkey -> sids (tautomer/stereo duplicates share one metric key)
        self.mkey2sids = {}
        for s, k in enumerate(self.s_mkey):
            if k is not None:
                self.mkey2sids.setdefault(k, []).append(s)
        print(f'library: {len(self.sid):,} spectra / {len(st):,} structures, {len(self.mz):,} cleaned peaks '
              f'({time.time()-t0:.0f}s)', flush=True)

    def spectra_of(self, sid):
        return self._by_sid[self._sid_bounds[sid]:self._sid_bounds[sid + 1]]

    def window(self, target, tol):
        lo = np.searchsorted(self.snm, target - tol, 'left')
        hi = np.searchsorted(self.snm, target + tol, 'right')
        return self.order[lo:hi]

    # ------------------------------------------------------------------ analog representatives
    def build_rep(self, prefer_instrument=None):
        """One representative spectrum per structure (prefer the test instrument, then the richest raw spectrum)."""
        pref = None
        if prefer_instrument:
            pref = str(prefer_instrument).lower().replace('-', '').replace(' ', '')
        self._pref = pref
        match = self._instr_match()
        self._match = match
        n = len(self.structs)
        # vectorised: sort by (sid, match desc, npk desc, index asc) and take first of each sid
        idx = np.arange(len(self.sid))
        keys = np.lexsort((idx, -self.npk.astype(np.int64), -match.astype(np.int64), self.sid))
        first = np.ones(len(keys), bool)
        ss = self.sid[keys]
        first[1:] = ss[1:] != ss[:-1]
        rep = np.sort(keys[first])
        nm = self.nm[rep]; ok = np.isfinite(nm)
        rep = rep[ok]; nm = nm[ok]
        o = np.argsort(nm)
        self.rep, self.rep_sid, self.rep_nm = rep[o], self.sid[rep[o]], nm[o]
        self.rep_of_sid = np.full(n, -1, np.int64)
        self.rep_of_sid[self.rep_sid] = self.rep
        print(f'  analog representatives: {len(self.rep):,} (instrument pref {prefer_instrument!r}: '
              f'{match.mean():.1%} of spectra match)', flush=True)

    def _instr_match(self):
        if self._pref is None:
            return np.zeros(len(self.sid), bool)
        u, inv = np.unique(np.asarray([x if isinstance(x, str) else '' for x in self.instrument], dtype=object),
                           return_inverse=True)
        um = np.array([x.lower().replace('-', '').replace(' ', '') == self._pref for x in u])
        return um[inv]

    def rep_excluding(self, sid, bad_spectra_mask_fn):
        """Best representative of `sid` among its spectra that survive a mask (same rule as build_rep)."""
        sp = self.spectra_of(sid)
        sp = sp[~bad_spectra_mask_fn(sp)]
        sp = sp[np.isfinite(self.nm[sp])]
        if len(sp) == 0:
            return -1
        o = np.lexsort((sp, -self.npk[sp].astype(np.int64), -self._match[sp].astype(np.int64)))
        return int(sp[o[0]])


In [ ]:
%%writefile /kaggle/working/casmi/metric.py
"""Official CASMI metric, re-implemented exactly (kaggle.com/code/metric/casmi-mean-reciprocal-rank).

key(smiles) = InChIKey(TautomerEnumerator().Canonicalize(MolFromSmiles(smiles)))[:14]  with RDKit 2026.03.3.
Score per molecule = 1/rank of the first guess whose key is in the accepted set, 0 if none within max_rank.
Empty entries (A;;B) are dropped before ranking; a trailing ';' is ignored.
"""
from __future__ import annotations

import warnings
from functools import lru_cache

import numpy as np

EXPECTED_RDKIT_VERSION = '2026.03.3'
_T = None


def _taut():
    global _T
    if _T is None:
        import rdkit
        from rdkit import RDLogger
        from rdkit.Chem.MolStandardize import rdMolStandardize
        RDLogger.DisableLog('rdApp.*')
        if rdkit.__version__ != EXPECTED_RDKIT_VERSION:
            warnings.warn(f'RDKit {rdkit.__version__} != metric {EXPECTED_RDKIT_VERSION}: tautomer keys may differ')
        _T = rdMolStandardize.TautomerEnumerator()
    return _T


@lru_cache(maxsize=2_000_000)
def score_key(smiles):
    """Tautomer-canonical InChIKey14, or None if RDKit cannot parse/canonicalize (== metric's canonicalize_smiles_to_ik14)."""
    if not isinstance(smiles, str):
        return None
    from rdkit import Chem
    te = _taut()
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        cm = te.Canonicalize(mol)
        ik = Chem.MolToInchiKey(cm)
        return ik[:14] if ik else None
    except (ValueError, RuntimeError, TypeError):
        return None


def score_keys_parallel(smiles_list, workers=8, chunksize=256):
    """Compute score keys for many SMILES with a process pool (order preserved)."""
    import multiprocessing as mp
    if workers <= 1 or len(smiles_list) < 2000:
        return [score_key(s) for s in smiles_list]
    ctx = mp.get_context('spawn')
    with ctx.Pool(workers) as p:
        return p.map(_score_key_plain, smiles_list, chunksize=chunksize)


def _score_key_plain(s):
    return score_key(s)


def parse_guesses(raw):
    return [c.strip() for c in str(raw).split(';') if c.strip()]


def reciprocal_rank(guesses, accepted_keys, max_rank=25):
    for r, g in enumerate(guesses[:max_rank], 1):
        if score_key(g) in accepted_keys:
            return 1.0 / r
    return 0.0


def mrr(solution: dict, submission: dict, max_rank=25):
    """solution: {molecule_id: answer smiles (comma-separated alternatives allowed)};
       submission: {molecule_id: 'smi1;smi2;...' or list}. Missing molecules score 0."""
    rr = []
    for mid, ans in solution.items():
        acc = {k for k in map(score_key, [s for s in ans.split(',') if s]) if k}
        if not acc:
            raise ValueError(f'unparseable solution for {mid}: {ans!r}')
        g = submission.get(mid, [])
        if isinstance(g, str):
            g = parse_guesses(g)
        if len(g) > max_rank:
            raise ValueError(f'{mid}: {len(g)} guesses > {max_rank}')
        rr.append(reciprocal_rank(g, acc, max_rank))
    return float(np.mean(rr)) if rr else 0.0


def rank_of_key(ranked_keys, truth_key, max_rank=25):
    """1-based rank of truth among already-keyed guesses, or 0 if absent (fast path for CV)."""
    for r, k in enumerate(ranked_keys[:max_rank], 1):
        if k == truth_key:
            return r
    return 0


In [ ]:
%%writefile /kaggle/working/casmi/prior.py
"""Candidate prior features ("how natural-product-like / well-known is this structure"), additive to the 31 features.

Structure-intrinsic descriptors only (NP-likeness, SA score, complexity, rings, sugars, element counts) plus
window-relative versions (z-score / rank inside the candidate window) and same-formula group statistics.
Pool provenance (COCONUT vs train-only) is available as a SEPARATE opt-in column group ('coco') because in the CV
simulation every answer is a training-library structure -- see research/04-pipeline-analysis.md.

Used by cvx.py (CV on cached feats). Per-SMILES descriptors are cached in outputs/cv/prior_desc.pkl.
"""
from __future__ import annotations

import multiprocessing as mp
import os
import pickle
import sys

import numpy as np

from . import config

DESC_NAMES = ['np_score', 'sa_score', 'heavy', 'rings', 'arom_rings', 'fsp3', 'stereo', 'bertz', 'nO', 'nN',
              'nS_P_hal', 'hbd', 'logp', 'sugars', 'methoxy', 'charge_abs']
_NP = _SA = None


def _init():
    global _NP, _SA
    from rdkit import RDLogger
    from rdkit.Chem import RDConfig
    RDLogger.DisableLog('rdApp.*')
    sys.path.append(os.path.join(RDConfig.RDContribDir, 'NP_Score'))
    sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
    import npscorer
    import sascorer
    _NP = (npscorer, npscorer.readNPModel())
    _SA = sascorer
    sascorer.readFragmentScores()


def _sugars(mol):
    ri = mol.GetRingInfo()
    n = 0
    for ring in ri.AtomRings():
        if len(ring) not in (5, 6):
            continue
        at = [mol.GetAtomWithIdx(i) for i in ring]
        if sum(a.GetSymbol() == 'O' for a in at) != 1 or sum(a.GetSymbol() == 'C' for a in at) != len(ring) - 1:
            continue
        if any(a.GetIsAromatic() for a in at):
            continue
        rs = set(ring)
        exo_o = 0
        for a in at:
            if a.GetSymbol() != 'C':
                continue
            for nb in a.GetNeighbors():
                if nb.GetIdx() not in rs and nb.GetSymbol() == 'O':
                    exo_o += 1
        n += exo_o >= 2
    return n


def describe(smi):
    from rdkit import Chem
    from rdkit.Chem import Crippen, Descriptors, GraphDescriptors, Lipinski, rdMolDescriptors
    if _NP is None:
        _init()
    try:
        m = Chem.MolFromSmiles(smi)
        if m is None:
            return [np.nan] * len(DESC_NAMES) + [None]
        sym = [a.GetSymbol() for a in m.GetAtoms()]
        methoxy = len(m.GetSubstructMatches(Chem.MolFromSmarts('[CH3][OX2][#6]')))
        out = [
            _NP[0].scoreMol(m, _NP[1]), _SA.calculateScore(m), m.GetNumHeavyAtoms(),
            rdMolDescriptors.CalcNumRings(m), rdMolDescriptors.CalcNumAromaticRings(m),
            rdMolDescriptors.CalcFractionCSP3(m),
            len(Chem.FindMolChiralCenters(m, includeUnassigned=True, useLegacyImplementation=False)),
            GraphDescriptors.BertzCT(m), sym.count('O'), sym.count('N'),
            sum(s in ('S', 'P', 'F', 'Cl', 'Br', 'I') for s in sym), Lipinski.NumHDonors(m), Crippen.MolLogP(m),
            _sugars(m), methoxy, sum(abs(a.GetFormalCharge()) for a in m.GetAtoms()),
        ]
        return [float(x) for x in out] + [rdMolDescriptors.CalcMolFormula(m)]
    except Exception:
        return [np.nan] * len(DESC_NAMES) + [None]


def describe_many(smiles, workers=None, cache=config.OUTPUTS / 'cv' / 'prior_desc.pkl'):
    """{smiles: (desc vector float32, formula)} with an on-disk cache."""
    D = pickle.load(open(cache, 'rb')) if cache and cache.exists() else {}
    todo = sorted({s for s in smiles if s not in D})
    if todo:
        workers = workers or max(1, (os.cpu_count() or 4) - 1)
        with mp.get_context('spawn').Pool(workers, initializer=_init) as p:
            res = p.map(describe, todo, chunksize=200)
        for s, r in zip(todo, res):
            D[s] = (np.asarray(r[:-1], np.float32), r[-1])
        if cache:
            pickle.dump(D, open(cache, 'wb'))
    return D


def _rank(x):
    o = np.argsort(-x, kind='stable'); r = np.empty(len(x)); r[o] = np.arange(len(x)); return r / max(1, len(x) - 1)


def _z(x):
    s = x.std()
    return (x - x.mean()) / s if s > 1e-9 else np.zeros_like(x)


# column groups for ablation (names returned alongside)
GROUPS = ('desc', 'desc_rel', 'formula', 'coco')


def prior_features(smiles, mass, target, is_coco, D, groups=('desc', 'desc_rel', 'formula')):
    """Extra per-candidate features for one window. smiles/mass/is_coco are per candidate; target = neutral mass."""
    nc = len(smiles)
    desc = np.stack([D[s][0] for s in smiles]).astype(np.float32)
    desc = np.where(np.isfinite(desc), desc, 0.0)
    forms = [D[s][1] for s in smiles]
    cols, names = [], []
    if 'desc' in groups:
        cols.append(desc[:, :-1]); names += DESC_NAMES[:-1]  # charge_abs dropped (COCONUT is charge-filtered)
    if 'desc_rel' in groups:
        for j, nm in enumerate(DESC_NAMES[:2] + ['bertz', 'sugars', 'heavy']):
            x = desc[:, DESC_NAMES.index(nm)]
            cols += [_z(x)[:, None], _rank(x)[:, None]]; names += [f'{nm}_z', f'{nm}_rank']
    if 'formula' in groups:
        from collections import Counter
        cnt = Counter(forms)
        same = np.array([cnt[f] for f in forms], np.float32)
        ppm = (np.asarray(mass) - target) / target * 1e6
        cols += [np.log(same)[:, None], (same / nc)[:, None], np.abs(ppm)[:, None], _rank(-np.abs(ppm))[:, None],
                 np.full((nc, 1), len(cnt), np.float32)]
        names += ['log_same_formula', 'frac_same_formula', 'abs_ppm', 'ppm_rank', 'n_formulas']
    if 'coco' in groups:
        c = np.asarray(is_coco, np.float32)
        cols += [c[:, None], np.full((nc, 1), c.mean(), np.float32)]; names += ['is_coco', 'coco_frac']
    return np.hstack(cols).astype(np.float32), names


In [ ]:
%%writefile /kaggle/working/casmi/ranker.py
"""Calibrated ranker: HistGradientBoosting P(candidate is the answer), bagged over class priors x seeds.

Training rows come from a class-1 simulation (M=0) and a class-2 simulation (M=1); sample weight w1 on M=0 rows and
1-w1 on M=1 rows, averaged over CFG.W1_PRIORS x CFG.SEEDS (the notebook's (0.30, 0.60) x (0,1,2,3)).
"""
from __future__ import annotations

import time

import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier

from . import config


class Ranker:
    def __init__(self, cfg: config.CFG, priors=None, seeds=None):
        self.cfg = cfg
        self.priors = tuple(priors if priors is not None else cfg.W1_PRIORS)
        self.seeds = tuple(seeds if seeds is not None else cfg.SEEDS)
        self.models = []

    def fit(self, X, Y, M, sw=None):
        """sw: optional per-row weight multiplier (e.g. up-weight np-examples molecules)."""
        t0 = time.time()
        self.models = []
        for w1 in self.priors:
            W = np.where(M == 0, w1, 1.0 - w1)
            if sw is not None:
                W = W * sw
            for sd in self.seeds:
                m = HistGradientBoostingClassifier(random_state=sd, **self.cfg.GBM)
                m.fit(X, Y, sample_weight=W)
                self.models.append(m)
        self.fit_secs = time.time() - t0
        return self

    def predict(self, X):
        return np.mean([m.predict_proba(X)[:, 1] for m in self.models], axis=0)

    @classmethod
    def from_rank_train(cls, cfg: config.CFG, path=config.RANK_TRAIN):
        """The notebook's shipped simulation rows (819 query groups; identities unknown -> not usable inside CV)."""
        z = np.load(path)
        r = cls(cfg).fit(z['X'], z['Y'], z['M'])
        print(f'ranker (rank_train.npz): {len(r.models)} GBMs on {len(z["Y"]):,} rows ({r.fit_secs:.0f}s)', flush=True)
        return r


In [ ]:
%%writefile /kaggle/working/casmi/rows_v2.py
"""Ranker v2: leak-free training rows from ALL 1000 holdout_v1 molecules + extra features + HGB/LambdaRank blend.

  uv run python -m casmi.rows_v2 build --fp-dir data/ext/casmi26-fp-models-h1      # -> outputs/ranker_rows_v2/
  DYLD_LIBRARY_PATH=.venv/lib/python3.12/site-packages/torch/lib uv run python -m casmi.rows_v2 eval
  uv run casmi predict --ranker v2 --ranker-rows outputs/ranker_rows_v2 --fp-dir data/ext/casmi26-fp-models-h1

Rows: every holdout_v1 molecule (cv 500 + extra 500; all excluded from the h1 FPNet's spectra and decoys) is
queried as in cv.py -- split A (class-1 sim, M=0; only A-eligible molecules) and split B (class-2 sim, M=1).
Features = the 31 base columns + prior (desc, desc_rel, formula; prior.py) + FP-family block (per-family f.z
z/rank/gap, confident-bit hinge counts, cosine; the cvx.fp_block columns). COCONUT membership is NEVER a feature.

Files in the rows dir:
  rows.npz   X (float32, n x d), Y, M (0=A, 1=B), G (group id = molecule*2 + split), mol (holdout index)
  meta.json  feature names, groups, fp weights used, counts
  keys.pkl   per-row metric keys + molecule info (evaluation only; not needed at predict time)
"""
from __future__ import annotations

import argparse
import json
import pickle
import time
from pathlib import Path

import numpy as np

from . import config
from .features import FEATURE_NAMES, _rank_norm, _z

NP_LIB = 'enveda-np-examples'
PRIOR_GROUPS = ('desc', 'desc_rel', 'formula')
FPB_NAMES = [f'{fam}_{k}' for fam in ('ecfp4', 'ecfp6', 'rdk', 'maccs') for k in ('z', 'rank', 'gap')] + \
            ['fpb_miss', 'fpb_extra', 'fpb_hinge_rank', 'fpb_cos_z', 'fpb_cos_rank', 'fpb_log_nvar']
DEFAULT_DIR = config.OUTPUTS / 'ranker_rows_v2'


# --------------------------------------------------------------------------------------------- extra features
def fp_family_block(F, z, blk):
    """= cvx.fp_block for one window. F: (nc, nbits) 0/1 candidate fps, z: FPNet logits (or None)."""
    nc = F.shape[0]
    if z is None:
        return np.zeros((nc, len(FPB_NAMES)), np.float32)
    F = F.astype(np.float32)
    ex = []
    for b in range(4):
        m = blk == b
        x = F[:, m] @ z[m]
        ex += [_z(x), _rank_norm(x), x - x.max()]
    mean = F.mean(0)
    var = (mean > 0) & (mean < 1)
    pr = 1 / (1 + np.exp(-z[var])); Fv = F[:, var]
    nv = max(int(var.sum()), 1)
    miss = ((1 - Fv) * (pr > 0.9)).sum(1); extra = (Fv * (pr < 0.1)).sum(1)
    cos = (Fv @ pr) / (np.linalg.norm(Fv, axis=1) * np.linalg.norm(pr) + 1e-9)
    ex += [miss, extra, _rank_norm(-(miss + extra)), _z(cos), _rank_norm(cos), np.full(nc, np.log(nv))]
    return np.column_stack(ex).astype(np.float32)


def family_of_bits():
    bits = np.load(config.COCO_DIR / 'fp_bits.npy')
    return np.searchsorted([4096, 8192, 10240], bits, 'right')  # 0 ECFP4, 1 ECFP6, 2 RDKit path, 3 MACCS


def extra_features(pool, pidx, target, z, D, blk):
    """Prior + FP-family columns for one candidate window (pool indices pidx). D = prior.describe_many dict."""
    from .prior import prior_features
    smis = [pool.smiles[i] for i in pidx]
    pf, names = prior_features(smis, pool.mass[pidx], target, np.zeros(len(pidx)), D, PRIOR_GROUPS)
    fb = fp_family_block(pool.fps(pidx), z, blk)
    return np.hstack([pf, fb]).astype(np.float32), names + FPB_NAMES


# --------------------------------------------------------------------------------------------- build rows
def build_rows(E, out_dir=DEFAULT_DIR, log_every=25, limit=None):
    from .cv import fixed_holdout, load_query_rows, make_query
    from .engine import Mask
    from .metric import score_key
    from .prior import describe_many
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    L, P = E.L, E.pool
    hold = fixed_holdout('all')[:limit]
    raw = load_query_rows([i for h in hold for i in h['q_rows']])
    t0 = time.time()
    part = out_dir / 'windows.pkl'   # resumable: per-molecule windows (base X, pidx, z)
    W = pickle.load(open(part, 'rb')) if part.exists() else []
    for j in range(len(W), len(hold)):
        h = hold[j]
        q = make_query(L, h, raw)
        w = dict(target=None, z=None, A=None, B=None)
        if q is not None:
            G = np.array(h['G'], np.int64)
            z = E.logits(q)
            mB = Mask(excl_sids=G)
            cB = E.candidates(q, E.lib_sim(q, mB), E.analog_sim(q, mB), z)
            w.update(target=q.target, z=z)
            if cB is not None:
                w['B'] = dict(X=cB['X'], pidx=cB['pidx'])
            if h['eligible_A']:
                mA = Mask(src_sids=G, src_lib=h['src_lib'])
                ov = {int(s): L.rep_excluding(int(s), lambda idx: mA.spectra_bad(L, idx)) for s in G}
                cA = E.candidates(q, E.lib_sim(q, mA), E.analog_sim(q, mA, ov), z)
                if cA is not None:
                    w['A'] = dict(X=cA['X'], pidx=cA['pidx'])
        W.append(w)
        if (j + 1) % log_every == 0 or j + 1 == len(hold):
            el = time.time() - t0
            print(f'  rows {j+1}/{len(hold)}  {el:.0f}s ({el/(j+1-0):.2f}s/mol)', flush=True)
            pickle.dump(W, open(part, 'wb'))
    t_win = time.time() - t0
    # metric keys + descriptors for every candidate
    allp = np.unique(np.concatenate([w[s]['pidx'] for w in W for s in 'AB' if w[s] is not None]))
    from .metric import score_keys_parallel
    keys = dict(zip(allp.tolist(), score_keys_parallel([P.smiles[i] for i in allp], workers=E.cfg.WORKERS)))
    t1 = time.time()
    D = describe_many([P.smiles[i] for i in allp], cache=out_dir / 'desc_cache.pkl')
    t_desc = time.time() - t1
    blk = family_of_bits()
    X, Y, M, Gr, MOL, KEYS = [], [], [], [], [], []
    names = None
    for j, (h, w) in enumerate(zip(hold, W)):
        for si, s in enumerate('AB'):
            c = w[s]
            if c is None:
                continue
            ex, en = extra_features(P, c['pidx'], w['target'], w['z'], D, blk)
            names = names or FEATURE_NAMES + en
            k = [keys[int(i)] for i in c['pidx']]
            X.append(np.hstack([c['X'], ex])); Y.append(np.array([x == h['mkey'] for x in k], np.float32))
            M.append(np.full(len(k), si, np.int8)); Gr.append(np.full(len(k), 2 * j + si, np.int32))
            MOL.append(np.full(len(k), j, np.int32)); KEYS += k
    X = np.vstack(X); Y = np.concatenate(Y); M = np.concatenate(M); Gr = np.concatenate(Gr); MOL = np.concatenate(MOL)
    np.savez_compressed(out_dir / 'rows.npz', X=X, Y=Y, M=M, G=Gr, mol=MOL)
    pickle.dump(dict(keys=KEYS, mols=[dict(src_lib=h['src_lib'], eligible_A=h['eligible_A'], mkey=h['mkey'])
                                      for h in hold]), open(out_dir / 'keys.pkl', 'wb'))
    meta = dict(names=names, n_base=len(FEATURE_NAMES), prior_groups=list(PRIOR_GROUPS), fpblock=True,
                fp_dir=str(config.FP_MODEL_DIR), n_mol=len(hold), n_rows=int(len(Y)), n_A_groups=int(sum(w['A'] is not None for w in W)),
                n_B_groups=int(sum(w['B'] is not None for w in W)), secs_windows=round(t_win), secs_desc=round(t_desc))
    json.dump(meta, open(out_dir / 'meta.json', 'w'), indent=1)
    print(f'rows: {X.shape} -> {out_dir}; {meta}', flush=True)
    return meta


def load_rows(rows_dir=DEFAULT_DIR):
    rows_dir = Path(rows_dir)
    z = np.load(rows_dir / 'rows.npz')
    return {k: z[k] for k in z.files}, json.load(open(rows_dir / 'meta.json'))


# --------------------------------------------------------------------------------------------- ranker
class RankerV2:
    """HGB (Ranker) and/or LightGBM LambdaRank on v2 rows; kind in {'hgb', 'lgb', 'blend'}.
    `cols` = feature columns used ('base' = the 31, 'all' = base + prior + FP-family)."""

    def __init__(self, cfg, kind='blend', cols='all'):
        self.cfg, self.kind, self.cols = cfg, kind, cols
        self.extra = cols == 'all'

    def fit(self, X, Y, M, G):
        from .cvx import LGBRank
        from .ranker import Ranker
        t0 = time.time()
        self.hgb = Ranker(self.cfg).fit(X, Y, M) if self.kind in ('hgb', 'blend') else None
        self.lgb = LGBRank(self.cfg).fit(X, Y, M, G) if self.kind in ('lgb', 'blend') else None
        self.fit_secs = time.time() - t0
        return self

    def predict(self, X):
        if self.kind == 'hgb':
            return self.hgb.predict(X)
        if self.kind == 'lgb':
            return self.lgb.predict(X)
        return 0.5 * (1 - _rank_norm(self.hgb.predict(X))) + 0.5 * (1 - _rank_norm(self.lgb.predict(X)))

    @classmethod
    def from_rows(cls, cfg, rows_dir=DEFAULT_DIR, kind='blend', cols='all'):
        R, meta = load_rows(rows_dir)
        X = R['X'] if cols == 'all' else R['X'][:, :meta['n_base']]
        r = cls(cfg, kind, cols).fit(X, R['Y'], R['M'], R['G'])
        r.names = meta['names'] if cols == 'all' else meta['names'][:meta['n_base']]
        print(f'ranker v2 ({kind}, {cols}: {X.shape[1]} cols) on {len(R["Y"]):,} rows / {meta["n_mol"]} molecules '
              f'({r.fit_secs:.0f}s)', flush=True)
        return r

    # ---- test-time featurization (two-pass: descriptors for all test candidates are computed in one batch)
    def prepare(self, E, windows, cache=None):
        """windows: list of (q, c, z). Computes descriptors for every candidate SMILES once."""
        if not self.extra:
            return
        from .prior import describe_many
        t0 = time.time()
        smis = [E.pool.smiles[i] for _, c, _ in windows if c is not None for i in c['pidx']]
        self.D = describe_many(smis, cache=cache)
        self.blk = family_of_bits()
        self.prep_secs = time.time() - t0
        print(f'  v2 descriptors: {len(set(smis)):,} candidate SMILES ({self.prep_secs:.0f}s)', flush=True)

    def features(self, E, q, c, z):
        if not self.extra:
            return c['X']
        ex, _ = extra_features(E.pool, c['pidx'], q.target, z, self.D, self.blk)
        X = np.hstack([c['X'], ex])
        assert X.shape[1] == len(self.names), (X.shape, len(self.names))
        return X


# --------------------------------------------------------------------------------------------- evaluation
def _rr(score, keys, Y, topn=25):
    order = np.argsort(-score, kind='stable')
    seen, r = set(), 0
    for i in order:
        k = keys[i]
        if k is None or k in seen:
            continue
        seen.add(k); r += 1
        if Y[i] > 0:
            return 1.0 / r
        if r >= topn:
            break
    return 0.0


def evaluate(rows_dir=DEFAULT_DIR, configs=('rank_train', 'hgb31', 'lgb31', 'blend31', 'hgb', 'lgb', 'blend'),
             n_folds=5, seed=0, out_name='cv_rr.npz', np_weight=1.0):
    """np_weight: training-row weight multiplier for enveda-np-examples molecules (e180 = 1). The 1000 rows are 250 np +
    750 e180 and e180 is much easier, while the test looks more np-like -> check whether matching that helps np CV."""
    from .cvx import LGBRank
    from .ranker import Ranker
    cfg = config.CFG()
    rows_dir = Path(rows_dir)
    R, meta = load_rows(rows_dir)
    K = pickle.load(open(rows_dir / 'keys.pkl', 'rb'))
    keys = np.asarray(K['keys'], dtype=object)
    nm = meta['n_mol']; nb = meta['n_base']
    X, Y, M, G, mol = R['X'], R['Y'], R['M'], R['G'], R['mol']
    fold = np.random.default_rng(seed).permutation(nm) % n_folds
    libs = np.array([m['src_lib'] for m in K['mols']])
    sw = np.where(libs[mol] == NP_LIB, np_weight, 1.0) if np_weight != 1.0 else None
    grp_idx = {g: np.where(G == g)[0] for g in np.unique(G)}
    res = {}
    prev = dict(np.load(rows_dir / out_name)) if (rows_dir / out_name).exists() else {}
    if 'fold' in prev and not np.array_equal(prev['fold'], fold):
        prev = {}
    todo = [c for c in configs if c not in prev]   # resumable: finished configs are kept

    def save():
        np.savez(rows_dir / out_name, **{**prev, **res, 'fold': fold})

    if 'rank_train' in todo:
        t0 = time.time()
        rk = Ranker.from_rank_train(cfg)
        p = rk.predict(X[:, :nb])
        rr = np.full((2, nm), np.nan)
        for g, ix in grp_idx.items():
            rr[g % 2][g // 2] = _rr(p[ix], keys[ix], Y[ix])
        res['rank_train'] = rr; save()
        print(f'  [rank_train] {time.time()-t0:.0f}s', flush=True)
    for cols in ('31', ''):
        fam = [c for c in ('hgb', 'lgb', 'blend') if c + cols in todo]
        if not fam:
            continue
        Xc = X[:, :nb] if cols == '31' else X
        rr = {c: np.full((2, nm), np.nan) for c in fam}
        t0 = time.time()
        need_h = 'hgb' in fam or 'blend' in fam; need_l = 'lgb' in fam or 'blend' in fam
        for f in range(n_folds):
            tr = fold[mol] != f
            swt = sw[tr] if sw is not None else None
            h = Ranker(cfg).fit(Xc[tr], Y[tr], M[tr], swt) if need_h else None
            l = LGBRank(cfg).fit(Xc[tr], Y[tr], M[tr], G[tr], swt) if need_l else None
            ph = h.predict(Xc) if h else None      # all rows at once (per-call overhead dominates otherwise)
            pl_ = l.predict(Xc) if l else None
            for g, ix in grp_idx.items():
                if fold[g // 2] != f:
                    continue
                for c in fam:
                    if c == 'blend':
                        p = 0.5 * (1 - _rank_norm(ph[ix])) + 0.5 * (1 - _rank_norm(pl_[ix]))
                    else:
                        p = (ph if c == 'hgb' else pl_)[ix]
                    rr[c][g % 2][g // 2] = _rr(p, keys[ix], Y[ix])
            print(f'  [{"/".join(fam)}{cols}] fold {f} done ({time.time()-t0:.0f}s)', flush=True)
        for c in fam:
            res[c + cols] = rr[c]
        save()
    save()
    return report(rows_dir, out_name)


def _stat(rr, mask, elig):
    """weighted = 0.16*mean(A over A-eligible) + 0.45*mean(B); split C (= 0 by construction) contributes 0."""
    A = rr[0][mask & elig]; B = rr[1][mask]
    return 0.16 * np.nanmean(A) + 0.45 * np.nanmean(B), np.nanmean(A), np.nanmean(B)


def report(rows_dir=DEFAULT_DIR, out_name='cv_rr.npz', base='rank_train', n_boot=2000):
    rows_dir = Path(rows_dir)
    Z = dict(np.load(rows_dir / out_name))
    # also show variants from other runs on the same folds (e.g. cv_rr_npw3.npz -> 'blend@npw3')
    for f in sorted(rows_dir.glob(Path(out_name).stem + '_*.npz')):
        V = dict(np.load(f))
        if np.array_equal(V.get('fold'), Z['fold']):
            Z.update({f'{k}@{f.stem.split("_", 2)[-1]}': v for k, v in V.items() if k != 'fold'})
    K = pickle.load(open(rows_dir / 'keys.pkl', 'rb'))
    libs = np.array([m['src_lib'] for m in K['mols']]); elig = np.array([m['eligible_A'] for m in K['mols']])
    # A for groups with no candidates / missing window -> 0 (as in cv.py)
    for k in Z:
        if k != 'fold':
            Z[k] = Z[k].copy(); Z[k][0][elig & np.isnan(Z[k][0])] = 0.0; Z[k][1][np.isnan(Z[k][1])] = 0.0
    npm = libs == 'enveda-np-examples'
    rng = np.random.default_rng(1)
    boots = {name: [rng.integers(0, m.sum(), m.sum()) for _ in range(n_boot)] for name, m in (('np', npm), ('all', np.ones_like(npm)))}
    lines = []
    for k, rr in Z.items():
        if k == 'fold':
            continue
        parts = []
        for name, m in (('np', npm), ('all', np.ones_like(npm))):
            w, a, b = _stat(rr, m, elig)
            s = f'{name}: W={w:.4f} A={a:.4f} B={b:.4f}'
            if base in Z and k != base:
                idx = np.where(m)[0]
                d = []
                for bi in boots[name]:
                    ii = idx[bi]; mm = np.zeros_like(m); np.add.at(mm, ii, 1)
                    # weighted statistic with multiplicity weights
                    wA = mm * elig; wB = mm
                    fa = lambda r: np.nansum(r[0] * wA) / wA.sum(); fb = lambda r: np.sum(r[1] * wB) / wB.sum()
                    d.append(0.16 * (fa(rr) - fa(Z[base])) + 0.45 * (fb(rr) - fb(Z[base])))
                d = np.array(d)
                s += f' d={w - _stat(Z[base], m, elig)[0]:+.4f} (se {d.std():.4f}, P>0 {np.mean(d > 0):.2f})'
            parts.append(s)
        lines.append(f'{k:>16} | ' + ' | '.join(parts))
    lines.append(f'n: np={npm.sum()} all={len(npm)} A-eligible np={int((elig & npm).sum())} all={int(elig.sum())}')
    txt = '\n'.join(lines)
    print(txt, flush=True)
    (rows_dir / 'cv_summary.txt').write_text(txt + '\n')
    return txt


def main(argv=None):
    ap = argparse.ArgumentParser('casmi.rows_v2')
    sub = ap.add_subparsers(dest='cmd', required=True)
    b = sub.add_parser('build'); b.add_argument('--out', default=str(DEFAULT_DIR))
    b.add_argument('--fp-dir', default=str(config.EXT / 'casmi26-fp-models-h1')); b.add_argument('--workers', type=int, default=None)
    b.add_argument('--limit', type=int, default=None)
    e = sub.add_parser('eval'); e.add_argument('--rows', default=str(DEFAULT_DIR))
    e.add_argument('--configs', default='rank_train,hgb31,lgb31,blend31,hgb,lgb,blend'); e.add_argument('--fold-seed', type=int, default=0)
    e.add_argument('--np-weight', type=float, default=1.0)
    r = sub.add_parser('report'); r.add_argument('--rows', default=str(DEFAULT_DIR))
    r.add_argument('--base', default='rank_train', help='row every other config is compared against')
    a = ap.parse_args(argv)
    if a.cmd == 'build':
        from .cli import _engine
        config.FP_MODEL_DIR = Path(a.fp_dir)
        cfg = config.CFG()
        if a.workers: cfg.WORKERS = a.workers
        E = _engine(cfg)
        try:
            build_rows(E, a.out, limit=a.limit)
        finally:
            if E.frag is not None: E.frag.close()
    elif a.cmd == 'eval':
        out = 'cv_rr' + (f'_s{a.fold_seed}' if a.fold_seed else '') + (f'_npw{a.np_weight:g}' if a.np_weight != 1.0 else '')
        evaluate(a.rows, tuple(a.configs.split(',')), seed=a.fold_seed, out_name=out + '.npz', np_weight=a.np_weight)
    else:
        report(a.rows, base=a.base)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/casmi/sim.py
"""Spectrum-simulation isomer re-scoring (ICEBERG, MassSpecGym `msg_all` weights) -- runtime helpers.

The simulator itself runs in a SEPARATE interpreter (RDKit 2025.03 is required by ICEBERG's fragment engine; the main
project pins RDKit 2026.03 for the metric). We reuse the public offline package `ahmedberatozer/casmi26-iceberg`
(data/ext/casmi26-iceberg: ice_runner.py + ms_pred MIT subset + pure-torch DGL shim + stripped MSG checkpoints):

    local : .venv-ice/bin/python data/ext/casmi26-iceberg/ice_runner.py PKG in.json out.json --device cpu --site none
    kaggle: python ice_runner.py PKG in.json out.json --device cuda --site /kaggle/working/ice_site  (installs wheels)

Score (runner definition, `ice_ex_mean`): for each covered query spectrum (positive [M+H]+ / [M+Na]+) ICEBERG predicts
the candidate at the query's adduct and 5-eV CE bucket (multi-CE / missing CE -> 20+40+60 merged), then entropy
similarity (0.01 Da / 20 ppm) query vs prediction; mean over the molecule's covered spectra. None = not covered.

This module is pure python/numpy (no torch / rdkit) so the main pipeline can import it:
  items_from_specs()  -> runner input
  run_runner()        -> {mid: {smiles: score|None}}
  fuse_formula_groups -> v4f post-ranker re-rank (z(ranker) + lam z(sim) inside same-formula groups, slots kept)
  fuse_rank_blend     -> alternative: (1-w) rank(ranker) + w rank(sim) inside the top-K
  sim_features        -> per-candidate columns for a ranker (raw, z / rank within formula group, gap to group max)
"""
from __future__ import annotations

import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[2]
ICE_PKG = ROOT / 'data' / 'ext' / 'casmi26-iceberg'
ICE_PY = ROOT / '.venv-ice' / 'bin' / 'python'
COV = ('[M+H]+', '[M+Na]+')


# --------------------------------------------------------------------------------------------- runner I/O
def spec_item(sp):
    """Our query-spectrum dict (cv.load_query_rows / engine.Query style: mz, it, prec, adduct, mode(+-1), instrument,
    ce (list of eV or None)) -> runner spectrum dict."""
    ce = sp.get('ce')
    ce = [] if ce is None else [float(v) for v in np.atleast_1d(np.asarray(ce, float)) if np.isfinite(v)]
    mode = sp.get('mode')
    mode = 'positive' if (mode == 1 or mode == 1.0 or mode == 'positive') else 'negative'
    return dict(mz=[float(v) for v in sp['mz']], it=[float(v) for v in sp['it']], prec=float(sp['prec']),
                adduct=sp.get('adduct'), mode=mode, instrument=sp.get('instrument'), ce_ev=ce, ce_orig=None,
                ce_units='eV')


def covered(sp):
    m = sp.get('mode')
    return (m == 1 or m == 1.0 or m == 'positive') and sp.get('adduct') in COV


def items_from_specs(mol_specs, mol_cands):
    """mol_specs: {mid: [spectrum dict]}, mol_cands: {mid: [smiles]} (priority order = dict order)."""
    out = []
    for mid, cands in mol_cands.items():
        sp = [spec_item(s) for s in mol_specs[mid] if covered(s)]
        if sp and cands:
            out.append(dict(mid=str(mid), spectra=sp, cands=list(dict.fromkeys(cands))))
    return out


def run_runner(items, workdir, device='cpu', budget_s=0.0, site='none', python=None, pkg=ICE_PKG, threads=0,
               extra=()):
    """Blocking subprocess run; returns ({mid: {smiles: score|None}}, meta)."""
    workdir = Path(workdir); workdir.mkdir(parents=True, exist_ok=True)
    fin, fout = workdir / 'ice_in.json', workdir / 'ice_out.json'
    json.dump(items, open(fin, 'w'))
    cmd = [str(python or ICE_PY), str(Path(pkg) / 'ice_runner.py'), str(pkg), str(fin), str(fout), '--device', device,
           '--budget', str(float(budget_s)), '--site', str(site), '--quiet']
    if threads:
        cmd += ['--threads', str(threads)]
    cmd += list(extra)
    subprocess.run(cmd, check=False)
    out = json.load(open(fout)) if fout.exists() else {}
    meta = json.load(open(str(fout) + '.meta.json')) if Path(str(fout) + '.meta.json').exists() else {}
    return out, meta


# --------------------------------------------------------------------------------------------- fusion
def _z(vals):
    """pandas-style group z (ddof=1) over finite members; missing / n<2 / sd 0 -> 0 (= fuse._z in casmi26-iceberg)."""
    v = np.asarray([np.nan if x is None else float(x) for x in vals], float)
    ok = np.isfinite(v)
    out = np.zeros(len(v))
    if ok.sum() < 2:
        return out
    mu = v[ok].mean(); sd = v[ok].std(ddof=1)
    if not sd > 0:
        return out
    out[ok] = (v[ok] - mu) / sd
    return out


def fuse_formula_groups(scores, sims, formulas, lam=0.5, top_n=60, min_covered=2):
    """v4f re-rank. scores/sims/formulas aligned with the CURRENT ranked order (index 0 = best). Returns a permutation
    (list of indices). Each same-formula group inside the first top_n positions with >= min_covered finite sims is
    re-sorted by z(score) + lam*z(sim) and put back into the same slots."""
    n = len(scores)
    order = list(range(n))
    m = min(int(top_n), n)
    groups = {}
    for i in range(m):
        groups.setdefault(formulas[i], []).append(i)
    for idx in groups.values():
        if len(idx) < 2:
            continue
        s = [sims[i] for i in idx]
        if sum(x is not None and math.isfinite(x) for x in s) < max(2, min_covered):
            continue
        fused = _z([scores[i] for i in idx]) + lam * _z(s)
        new = [idx[k] for k in sorted(range(len(idx)), key=lambda k: (-fused[k], k))]
        for slot, src in zip(idx, new):
            order[slot] = src
    return order


def fuse_rank_blend(scores, sims, w=0.5, top_n=60):
    """Global alternative: inside the first top_n (ranked order), fused = (1-w) * rank-score(ranker) + w *
    rank-score(sim) (rank-score in [0,1], 1 = best; missing sim -> 0.5). Positions > top_n untouched."""
    n = len(scores); m = min(int(top_n), n)
    if m < 2:
        return list(range(n))
    rs = 1.0 - np.arange(m) / (m - 1)
    s = np.array([np.nan if x is None else float(x) for x in sims[:m]])
    ok = np.isfinite(s)
    rq = np.full(m, 0.5)
    if ok.sum() >= 2:
        o = np.argsort(-s[ok], kind='stable')
        r = np.empty(ok.sum()); r[o] = 1.0 - np.arange(ok.sum()) / (ok.sum() - 1)
        rq[ok] = r
    f = (1 - w) * rs + w * rq
    top = sorted(range(m), key=lambda k: (-f[k], k))
    return top + list(range(m, n))


def sim_features(sims, formulas):
    """Per-candidate ranker columns: raw sim (NaN->-1), covered flag, z within formula group, rank-norm within
    formula group (1 = best), sim - group max. For later wiring into the ranker (not done here)."""
    s = np.array([np.nan if x is None else float(x) for x in sims])
    f = np.asarray(formulas, object)
    zc = np.zeros(len(s)); rk = np.zeros(len(s)); gap = np.zeros(len(s))
    for g in set(f.tolist()):
        idx = np.where(f == g)[0]
        zc[idx] = _z(s[idx])
        v = s[idx]; ok = np.isfinite(v)
        if ok.sum():
            o = np.argsort(-v[ok], kind='stable'); r = np.empty(ok.sum())
            r[o] = 1.0 - np.arange(ok.sum()) / max(ok.sum() - 1, 1)
            tmp = np.zeros(len(idx)); tmp[ok] = r; rk[idx] = tmp
            gp = np.zeros(len(idx)); gp[ok] = v[ok] - v[ok].max(); gap[idx] = gp
    return np.column_stack([np.nan_to_num(s, nan=-1.0), np.isfinite(s), zc, rk, gap]).astype(np.float32)


SIM_FEATURE_NAMES = ['ice_sim', 'ice_cov', 'ice_z_form', 'ice_rank_form', 'ice_gap_form']


In [ ]:
%%writefile /kaggle/working/casmi/sim_cv.py
"""CV of ICEBERG isomer re-scoring on the exp000 leak-free holdout (cached features / candidates; no channel recompute).

  uv run python -m casmi.sim_cv leak                 # MassSpecGym overlap of CV truths (+ top-K decoys)
  uv run python -m casmi.sim_cv oof --name base      # per-candidate out-of-fold ranker scores (cvx folds)
  uv run python -m casmi.sim_cv oof --name best --groups desc,desc_rel,formula --fpblock --ranker blend
  uv run python -m casmi.sim_cv jobs --k 60 --shards 6
  (run .venv-ice/bin/python data/ext/casmi26-iceberg/ice_runner.py per shard; see research/06-sim-rescoring.md)
  uv run python -m casmi.sim_cv eval --name best

Outputs under outputs/cv/exp000/sim/:
  msg_overlap.json    per CV molecule: truth metric key in MassSpecGym (any fold) / plain InChIKey14 match
  oof_<name>.pkl      [{split: scores array aligned with feats.pkl candidates}], fold
  formulas.pkl        {pool idx: formula}
  ice_in_<i>.json     runner shards;  ice_out_<i>.json runner outputs
  ice_scores.pkl      {mol index: {smiles: score|None}}   <- the per-(molecule, candidate) similarity cache
"""
from __future__ import annotations

import argparse
import json
import pickle
import time

import numpy as np

from . import config
from .cv import NP_LIB, WEIGHTS
from .sim import fuse_formula_groups, fuse_rank_blend, items_from_specs

OUT = config.OUTPUTS / 'cv'
MSG_TSV = config.EXT / 'massspecgym' / 'MassSpecGym.tsv'


def sdir(tag):
    d = OUT / tag / 'sim'
    d.mkdir(parents=True, exist_ok=True)
    return d


# --------------------------------------------------------------------------------------------- leakage
def _ik14_plain(smi):
    from rdkit import Chem
    m = Chem.MolFromSmiles(smi)
    return Chem.MolToInchiKey(m)[:14] if m is not None else None


def leak(tag='exp000'):
    import polars as pl
    from .metric import score_keys_parallel
    msg = pl.read_csv(MSG_TSV, separator='\t', columns=['smiles', 'inchikey', 'fold'])
    usmi = msg.unique('smiles')
    cache = sdir(tag) / 'msg_keys.pkl'
    if cache.exists():
        k_of = pickle.load(open(cache, 'rb'))
    else:
        sm = usmi['smiles'].to_list()
        k_of = dict(zip(sm, score_keys_parallel(sm, workers=8)))
        pickle.dump(k_of, open(cache, 'wb'))
    msg_mkeys = set(v for v in k_of.values() if v)
    msg_ik = set(msg['inchikey'].str.slice(0, 14).to_list())
    fold_of = {}
    for s, f in zip(usmi['smiles'].to_list(), usmi['fold'].to_list()):
        fold_of.setdefault(k_of.get(s), f)
    recs = pickle.load(open(OUT / tag / 'feats.pkl', 'rb'))
    cands = pickle.load(open(OUT / tag / 'cands.pkl', 'rb'))
    rows = []
    for j, r in enumerate(recs):
        h = r['h']
        ik = _ik14_plain(h['smiles'])
        rows.append(dict(j=j, lib=h['src_lib'], mkey=h['mkey'], in_msg=h['mkey'] in msg_mkeys,
                         in_msg_ik=ik in msg_ik, msg_fold=fold_of.get(h['mkey']), eligible_A=h['eligible_A']))
    # decoys: share of window candidates (split B) whose metric key is in MSG
    dec = []
    for r in recs:
        p = r['B']
        if p is None:
            continue
        ks = [k for k, y in zip(p['keys'], p['Y']) if y == 0]
        if ks:
            dec.append(np.mean([k in msg_mkeys for k in ks]))
    json.dump(rows, open(sdir(tag) / 'msg_overlap.json', 'w'), indent=0)
    for lib in (NP_LIB, 'enveda-180'):
        rr = [x for x in rows if x['lib'] == lib]
        print(f'{lib}: n={len(rr)} truth in MSG (metric key)={sum(x["in_msg"] for x in rr)} '
              f'(plain ik14 {sum(x["in_msg_ik"] for x in rr)}); among A-eligible '
              f'{sum(x["in_msg"] for x in rr if x["eligible_A"])}/{sum(x["eligible_A"] for x in rr)}; folds '
              f'{dict(zip(*np.unique([x["msg_fold"] for x in rr if x["in_msg"]], return_counts=True)))}')
    print(f'decoy share in MSG (split B window, mean over mols) = {np.mean(dec):.3f}; MSG unique keys {len(msg_mkeys)}')


# --------------------------------------------------------------------------------------------- OOF ranker scores
def oof(tag='exp000', name='base', groups='', fpblock=False, ranker='hgb', fold_seed=0):
    """Same folds / training rows as cvx.evaluate, but keeps per-candidate scores (higher = better)."""
    from . import cvx
    from .cv import rr_from_scores
    from .ranker import Ranker
    cfg = config.CFG()
    t0 = time.time()
    recs = pickle.load(open(OUT / tag / 'feats.pkl', 'rb'))
    if groups:
        recs, _ = cvx.augment(recs, pickle.load(open(OUT / tag / 'cands.pkl', 'rb')), tuple(groups.split(',')))
    if fpblock:
        recs = cvx.fp_block(recs, pickle.load(open(OUT / tag / 'cands.pkl', 'rb')), tag)
    rng = np.random.default_rng(fold_seed)
    fold = rng.permutation(len(recs)) % 5
    S = [dict() for _ in recs]
    rr = {s: np.full(len(recs), np.nan) for s in 'ABC'}
    for f in range(5):
        tr = [recs[i] for i in range(len(recs)) if fold[i] != f]
        parts = [cvx._stack(tr, 'A', 0), cvx._stack(tr, 'B', 10 ** 6)]
        X = np.vstack([x for p in parts for x in p[0]]); Y = np.concatenate([x for p in parts for x in p[1]])
        M = np.concatenate([x for p in parts for x in p[2]]); G = np.concatenate([x for p in parts for x in p[3]])
        if ranker == 'hgb':
            rk = Ranker(cfg).fit(X, Y, M)
        elif ranker == 'blend':
            rk = cvx.Blend(Ranker(cfg).fit(X, Y, M), cvx.LGBRank(cfg).fit(X, Y, M, G))
        else:
            rk = cvx.LGBRank(cfg).fit(X, Y, M, G)
        for i in np.where(fold == f)[0]:
            for s in 'ABC':
                p = recs[i][s]
                if p is None or (s == 'A' and not recs[i]['h']['eligible_A']):
                    continue
                S[i][s] = rk.predict(p['X']).astype(np.float64)
                rr[s][i] = rr_from_scores(p, S[i][s])
        print(f'  fold {f} done {time.time()-t0:.0f}s', flush=True)
    pickle.dump(dict(scores=S, fold=fold, rr=rr), open(sdir(tag) / f'oof_{name}.pkl', 'wb'))
    np_ = np.array([r['h']['src_lib'] == NP_LIB for r in recs])
    print(f'[{name}] A[np]={np.nanmean(rr["A"][np_]):.4f} B[np]={np.nanmean(rr["B"][np_]):.4f} '
          f'W[np]={(WEIGHTS["A"]*np.nan_to_num(rr["A"])+WEIGHTS["B"]*rr["B"])[np_].mean():.4f} '
          f'B[e180]={np.nanmean(rr["B"][~np_]):.4f} ({time.time()-t0:.0f}s)')


# --------------------------------------------------------------------------------------------- jobs
def _formulas(cands, tag):
    fp = sdir(tag) / 'formulas.pkl'
    if fp.exists():
        return pickle.load(open(fp, 'rb'))
    from rdkit import Chem
    from rdkit.Chem.rdMolDescriptors import CalcMolFormula
    F = {}
    for i, s in cands['smiles'].items():
        m = Chem.MolFromSmiles(s)
        F[i] = CalcMolFormula(m) if m is not None else f'?{i}'
    pickle.dump(F, open(fp, 'wb'))
    return F


def ranked(p, pid, score, k):
    """Ranked, metric-key-deduped list of (feats row index) -- exactly the submit order; first k."""
    order = np.argsort(-score, kind='stable')
    seen, out = set(), []
    for i in order:
        key = p['keys'][i]
        if key is None or key in seen or pid[i] < 0:
            continue
        seen.add(key); out.append(int(i))
        if len(out) >= k:
            break
    return out


def jobs(tag='exp000', k=60, shards=6, names=('base', 'best')):
    recs = pickle.load(open(OUT / tag / 'feats.pkl', 'rb'))
    cands = pickle.load(open(OUT / tag / 'cands.pkl', 'rb'))
    raw = pickle.load(open(OUT / tag / 'query_raw.pkl', 'rb'))
    O = [pickle.load(open(sdir(tag) / f'oof_{n}.pkl', 'rb')) for n in names if (sdir(tag) / f'oof_{n}.pkl').exists()]
    specs, mc = {}, {}
    for j, r in enumerate(recs):
        specs[j] = [raw[q] for q in r['h']['q_rows']]
        sm = []
        for o in O:
            for s in 'ABC':
                if s not in o['scores'][j]:
                    continue
                p = r[s]; pid = cands['pidx'][j][s]
                sm += [cands['smiles'][int(pid[i])] for i in ranked(p, pid, o['scores'][j][s], k)]
        mc[j] = list(dict.fromkeys(sm))
    items = items_from_specs(specs, mc)
    # balance shards by job count (candidates x distinct CE conditions)
    cost = [len(it['cands']) * len({(s['adduct'], tuple(s['ce_ev'])) for s in it['spectra']}) for it in items]
    buckets = [[] for _ in range(shards)]; load = np.zeros(shards)
    for i in np.argsort(-np.array(cost)):
        b = int(np.argmin(load)); buckets[b].append(items[i]); load[b] += cost[i]
    for b, its in enumerate(buckets):
        json.dump(its, open(sdir(tag) / f'ice_in_{b}.json', 'w'))
    print(f'{len(items)} covered molecules, {sum(len(it["cands"]) for it in items)} candidates, ~{sum(cost)} jobs '
          f'-> {shards} shards (load {load.astype(int).tolist()})')


def resume(tag='exp000'):
    """After an interrupted run: move finished molecules of ice_out_<i>.json into ice_done_<i>.json and rewrite
    ice_in_<i>.json with the molecules that still have no score (then relaunch the runners)."""
    import glob
    for fin in sorted(glob.glob(str(sdir(tag) / 'ice_in_*.json'))):
        i = fin.rsplit('_', 1)[1][:-5]
        fout = sdir(tag) / f'ice_out_{i}.json'; fdone = sdir(tag) / f'ice_done_{i}.json'
        done = json.load(open(fdone)) if fdone.exists() else {}
        if fout.exists():
            for mid, d in json.load(open(fout)).items():
                if any(v is not None for v in d.values()):
                    done[mid] = d
        json.dump(done, open(fdone, 'w'))
        todo = [it for it in json.load(open(fin)) if it['mid'] not in done]
        json.dump(todo, open(fin, 'w'))
        print(f'shard {i}: done {len(done)} todo {len(todo)}')


def collect(tag='exp000'):
    import glob
    S = {}
    for f in sorted(glob.glob(str(sdir(tag) / 'ice_done_*.json'))) + sorted(glob.glob(str(sdir(tag) / 'ice_out_*.json'))):
        for mid, d in json.load(open(f)).items():
            if int(mid) not in S or any(v is not None for v in d.values()):
                S[int(mid)] = d
    pickle.dump(S, open(sdir(tag) / 'ice_scores.pkl', 'wb'))
    n = sum(len(v) for v in S.values()); nn = sum(v is not None for d in S.values() for v in d.values())
    print(f'collected {len(S)} molecules, {nn}/{n} candidate scores')
    return S


# --------------------------------------------------------------------------------------------- evaluation
def _rr_from_order(keys_ranked, truth, topn=25):
    for r, key in enumerate(keys_ranked[:topn]):
        if key == truth:
            return 1.0 / (r + 1)
    return 0.0


def method_rr(recs, cands, F, S, O, j, split, method, param, k=60):
    """Reciprocal rank of molecule j / split after re-ranking the ranker's top-k with the sim scores."""
    r = recs[j]; p = r[split]
    if p is None:
        return 0.0
    pid = cands['pidx'][j][split]
    sc = O['scores'][j][split]
    lst = ranked(p, pid, sc, 10 ** 9)                    # full deduped ranked list
    keys = [p['keys'][i] for i in lst]
    ice = S.get(j) or {}
    if method != 'ranker' and ice:
        top = lst[:k]
        smis = [cands['smiles'][int(pid[i])] for i in top]
        sims = [ice.get(s) for s in smis]
        if method == 'formula':
            order = fuse_formula_groups([sc[i] for i in top], sims, [F[int(pid[i])] for i in top], lam=param, top_n=k)
        elif method == 'blend':
            order = fuse_rank_blend([sc[i] for i in top], sims, w=param, top_n=k)
        elif method == 'sim':                              # sim alone inside top-k (uncovered keep ranker order after)
            order = sorted(range(len(top)), key=lambda q: (-(sims[q] if sims[q] is not None else -1.0), q))
        keys = [keys[q] for q in order] + keys[k:]
    return _rr_from_order(keys, r['h']['mkey'])


def _perf(rrA, rrB):
    return WEIGHTS['A'] * np.nan_to_num(rrA) + WEIGHTS['B'] * rrB


def evaluate(tag='exp000', name='best', k=60, n_boot=2000):
    recs = pickle.load(open(OUT / tag / 'feats.pkl', 'rb'))
    cands = pickle.load(open(OUT / tag / 'cands.pkl', 'rb'))
    F = _formulas(cands, tag)
    O = pickle.load(open(sdir(tag) / f'oof_{name}.pkl', 'rb'))
    S = pickle.load(open(sdir(tag) / 'ice_scores.pkl', 'rb'))
    ov = {x['j']: x for x in json.load(open(sdir(tag) / 'msg_overlap.json'))}
    fold = O['fold']
    n = len(recs)
    lib = np.array([r['h']['src_lib'] for r in recs])
    inmsg = np.array([ov[j]['in_msg'] for j in range(n)])
    intr = np.array([ov[j]['in_msg'] and ov[j]['msg_fold'] == 'train' for j in range(n)])   # seen by msg_all ICEBERG
    cov = np.array([bool(S.get(j)) and any(v is not None for v in S[j].values()) for j in range(n)])
    eligA = np.array([r['h']['eligible_A'] for r in recs])

    def grid_rr(method, param):
        out = {}
        for s in 'AB':
            v = np.full(n, np.nan)
            for j in range(n):
                if s == 'A' and not eligA[j]:
                    continue
                if s not in O['scores'][j]:
                    v[j] = 0.0 if recs[j][s] is None else np.nan
                    continue
                v[j] = method_rr(recs, cands, F, S, O, j, s, method, param, k)
            out[s] = v
        return out

    base = grid_rr('ranker', 0)
    # sanity: equals the stored OOF rr
    assert np.allclose(np.nan_to_num(base['B']), np.nan_to_num(O['rr']['B'])), 'rank reproduction mismatch'
    grids = {'formula': [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0], 'blend': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8], 'sim': [0]}
    R = {(m, prm): grid_rr(m, prm) for m, ps in grids.items() for prm in ps}
    R[('ranker', 0)] = base
    pickle.dump(dict(R=R, fold=fold), open(sdir(tag) / f'eval_{name}_k{k}.pkl', 'wb'))

    def boot(d, rng=np.random.default_rng(1)):
        return np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)]).std()

    def nested(method, mask_fn):
        """Pick the param on the other 4 folds (W[np] over np molecules with sim coverage, excluding MSG-overlap
        molecules when mask_fn does), apply to the held-out fold."""
        ps = grids[method]
        out = {s: np.full(n, np.nan) for s in 'AB'}
        chosen = []
        for f in range(5):
            tr = (fold != f) & mask_fn
            best = max(ps, key=lambda q: np.nanmean(_perf(R[(method, q)]['A'], R[(method, q)]['B'])[tr]))
            chosen.append(best)
            te = fold == f
            for s in 'AB':
                out[s][te] = R[(method, best)][s][te]
        return out, chosen

    lines = [f'# ICEBERG re-rank on top-{k} of ranker "{name}" (exp000 holdout)']
    subsets = {
        'np all (n=250)': lib == NP_LIB,
        'np sim-covered': (lib == NP_LIB) & cov,
        'np not-in-MSG-train (clean)': (lib == NP_LIB) & ~intr,
        'np not-in-MSG-train covered': (lib == NP_LIB) & ~intr & cov,
        'np in-MSG-train (leaky)': (lib == NP_LIB) & intr,
        'non-np (e180 or msgfree) all': lib != NP_LIB,
        'non-np sim-covered': (lib != NP_LIB) & cov,
    }
    rows = [('ranker', 0)] + [(m, q) for m in ('formula', 'blend', 'sim') for q in grids[m]]
    for sname, msk in subsets.items():
        if msk.sum() == 0:
            continue
        lines.append(f'\n## {sname}: n={int(msk.sum())}')
        lines.append('| method | param | A | B | W(.16A+.45B) | dW vs ranker (se) | dB (se) |')
        lines.append('|---|---|---|---|---|---|---|')
        bw = _perf(base['A'], base['B'])
        for m, q in rows:
            rr = R[(m, q)]
            w = _perf(rr['A'], rr['B'])
            d = (w - bw)[msk]; dB = (rr['B'] - base['B'])[msk]
            lines.append(f'| {m} | {q} | {np.nanmean(rr["A"][msk]):.4f} | {np.nanmean(rr["B"][msk]):.4f} | '
                         f'{np.nanmean(w[msk]):.4f} | {np.nanmean(d):+.4f} ({boot(np.nan_to_num(d)):.4f}) | '
                         f'{np.nanmean(dB):+.4f} ({boot(np.nan_to_num(dB)):.4f}) |')
        for m in ('formula', 'blend'):
            for tune_name, tmask in (('tune np', lib == NP_LIB), ('tune non-np', lib != NP_LIB)):
                if tmask.sum() == 0:
                    continue
                nr, ch = nested(m, tmask)
                w = _perf(nr['A'], nr['B']); d = (w - bw)[msk]; dB = (nr['B'] - base['B'])[msk]
                lines.append(f'| nested {m} ({tune_name}) | {ch} | {np.nanmean(nr["A"][msk]):.4f} | '
                             f'{np.nanmean(nr["B"][msk]):.4f} | {np.nanmean(w[msk]):.4f} | {np.nanmean(d):+.4f} '
                             f'({boot(np.nan_to_num(d)):.4f}) | {np.nanmean(dB):+.4f} ({boot(np.nan_to_num(dB)):.4f}) |')
    txt = '\n'.join(lines)
    (sdir(tag) / f'eval_{name}_k{k}.md').write_text(txt + '\n')
    print(txt)


# --------------------------------------------------------------------------------------------- MSG-free holdout
def msgfree_holdout(n=300, seed=7, tag='exp000'):
    """Class-2-like holdout that neither simulator nor FPNet(h1) has seen: metric keys from holdout_v1 'fpval'
    (excluded from h1 FPNet training), NOT in MassSpecGym (any fold), with no timsTOF-library spectra; queried with
    k of its own spectra from one source library (k ~ test spectra-per-molecule). Split B only."""
    import polars as pl
    from .library import Library
    from .train_fpnet import load_holdout
    k_of = pickle.load(open(sdir(tag) / 'msg_keys.pkl', 'rb'))
    msg = set(v for v in k_of.values() if v)
    H = load_holdout(); fv = set(H['fpval']) - msg
    L = Library(config.CFG())
    te = pl.read_parquet(config.TEST, columns=['molecule_id'])
    k_dist = te.group_by('molecule_id').len()['len'].to_numpy()
    rng = np.random.default_rng(seed)
    keys = sorted(k for k in fv if k in L.mkey2sids)
    rng.shuffle(keys)
    out = []
    for mk in keys:
        G = np.array(sorted(L.mkey2sids[mk]), np.int64)
        sp = np.concatenate([L.spectra_of(g) for g in G])
        libs = set(L.lib[sp].tolist())
        if libs & {'enveda-180', NP_LIB}:
            continue
        pos = sp[(L.meta['adduct'].to_numpy()[sp] == '[M+H]+') & np.isfinite(L.nm[sp])]
        if not len(pos):
            continue
        lib = L.lib[pos[0]]
        q = sp[(L.lib[sp] == lib) & np.isfinite(L.nm[sp])]
        k = int(min(len(q), rng.choice(k_dist)))
        q = np.sort(rng.choice(q, size=k, replace=False))
        s0 = int(G[0])
        out.append(dict(sid=s0, mkey=mk, smiles=L.s_smiles[s0], src_lib=str(lib), G=G.tolist(), q_rows=q.tolist(),
                        eligible_A=False))
        if len(out) >= n:
            break
    d = OUT / 'msgfree'; d.mkdir(parents=True, exist_ok=True)
    json.dump(out, open(d / 'holdout.json', 'w'))
    from collections import Counter
    print(f'msgfree holdout {len(out)}: {Counter(h["src_lib"] for h in out)}')


def msgfree_feats():
    """Channel features for the msgfree holdout with the h1 FPNet (leak-free for these keys)."""
    from pathlib import Path
    from .cli import _engine
    from .cv import compute_features, load_query_rows
    config.FP_MODEL_DIR = config.EXT / 'casmi26-fp-models-h1'
    cfg = config.CFG()
    d = OUT / 'msgfree'
    hold = json.load(open(d / 'holdout.json'))
    E = _engine(cfg)
    raw = load_query_rows([i for h in hold for i in h['q_rows']])
    pickle.dump(raw, open(d / 'query_raw.pkl', 'wb'))
    recs, tim = compute_features(E, hold, raw)
    pickle.dump(recs, open(d / 'feats.pkl', 'wb'))
    if E.frag is not None:
        E.frag.close()


def msgfree_scores(train_tag='exp001-fph1', variant='full'):
    """Base HGB ranker (31 cols) fitted on ALL rows of train_tag (A as M=0, B as M=1) -> scores for msgfree split B
    (independent molecules, so no folds needed). variant noFP zeroes the FP columns in train and test."""
    from .cv import FP_COLS, cv_ranker, rr_from_scores
    cfg = config.CFG()
    cz = FP_COLS if variant == 'noFP' else None
    rk = cv_ranker(cfg, train_tag, cz)
    recs = pickle.load(open(OUT / 'msgfree' / 'feats.pkl', 'rb'))
    S, rr = [], {s: np.full(len(recs), np.nan) for s in 'ABC'}
    for j, r in enumerate(recs):
        d = {}
        for s in 'BC':
            p = r[s]
            if p is None:
                continue
            x = p['X'].copy()
            if cz:
                x[:, cz] = 0.0
            d[s] = rk.predict(x).astype(np.float64); rr[s][j] = rr_from_scores(p, d[s])
        S.append(d)
    fold = np.random.default_rng(0).permutation(len(recs)) % 5
    pickle.dump(dict(scores=S, fold=fold, rr=rr), open(sdir('msgfree') / f'oof_{variant}.pkl', 'wb'))
    print(f'[msgfree {variant}] B={np.nanmean(rr["B"]):.4f} n={int(np.isfinite(rr["B"]).sum())}')


def main(argv=None):
    ap = argparse.ArgumentParser('casmi.sim_cv')
    sub = ap.add_subparsers(dest='cmd', required=True)
    sub.add_parser('leak').add_argument('--tag', default='exp000')
    o = sub.add_parser('oof')
    o.add_argument('--tag', default='exp000'); o.add_argument('--name', required=True)
    o.add_argument('--groups', default=''); o.add_argument('--fpblock', action='store_true')
    o.add_argument('--ranker', default='hgb', choices=['hgb', 'lgb_rank', 'blend'])
    j = sub.add_parser('jobs'); j.add_argument('--tag', default='exp000'); j.add_argument('--k', type=int, default=60)
    j.add_argument('--shards', type=int, default=6)
    j.add_argument('--names', default='base,best')
    sub.add_parser('collect').add_argument('--tag', default='exp000')
    sub.add_parser('resume').add_argument('--tag', default='exp000')
    e = sub.add_parser('eval'); e.add_argument('--tag', default='exp000'); e.add_argument('--name', default='best')
    e.add_argument('--k', type=int, default=60)
    mh = sub.add_parser('msgfree'); mh.add_argument('step', choices=['holdout', 'feats', 'scores'])
    mh.add_argument('--variant', default='full')
    a = ap.parse_args(argv)
    if a.cmd == 'msgfree':
        {'holdout': msgfree_holdout, 'feats': msgfree_feats,
         'scores': lambda: msgfree_scores(variant=a.variant)}[a.step]()
    elif a.cmd == 'leak':
        leak(a.tag)
    elif a.cmd == 'oof':
        oof(a.tag, a.name, a.groups, a.fpblock, a.ranker)
    elif a.cmd == 'jobs':
        jobs(a.tag, a.k, a.shards, tuple(a.names.split(',')))
    elif a.cmd == 'resume':
        resume(a.tag)
    elif a.cmd == 'collect':
        collect(a.tag)
    elif a.cmd == 'eval':
        evaluate(a.tag, a.name, a.k)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/casmi/spectra.py
"""Similarity kernels: entropy-weighted spectral entropy similarity (Li et al. 2021) + mass-shifted variant.

Verbatim numba port of the notebook kernels. One change for speed that preserves results exactly:
library spectra are cleaned ONCE at cache-build time (clean_batch) instead of on every comparison,
so `search*` here take pre-cleaned library arrays.
"""
from __future__ import annotations

import numpy as np
from numba import njit, prange


@njit(cache=True, fastmath=True)
def _clean(mz, it, floor, topk, power, ent_weight):
    n = len(mz)
    if n == 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    mx = 0.0
    for i in range(n):
        if it[i] > mx:
            mx = it[i]
    if mx <= 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    thr = floor * mx
    c = 0
    for i in range(n):
        if it[i] >= thr:
            c += 1
    idx = np.empty(c, np.int64)
    j = 0
    for i in range(n):
        if it[i] >= thr:
            idx[j] = i
            j += 1
    if c > topk:
        v = np.empty(c, np.float32)
        for i in range(c):
            v[i] = it[idx[i]]
        o = np.argsort(v)[c - topk:]
        k2 = np.empty(topk, np.int64)
        for i in range(topk):
            k2[i] = idx[o[i]]
        k2.sort()
        idx = k2
        c = topk
    om = np.empty(c, np.float32)
    oi = np.empty(c, np.float32)
    s = 0.0
    for i in range(c):
        om[i] = mz[idx[i]]
        v = it[idx[i]] ** power
        oi[i] = v
        s += v
    if s > 0:
        for i in range(c):
            oi[i] /= s
    if ent_weight:
        S = 0.0
        for i in range(c):
            if oi[i] > 0:
                S -= oi[i] * np.log(oi[i])
        if S < 3.0:
            w = 0.25 + 0.25 * S
            s2 = 0.0
            for i in range(c):
                oi[i] = oi[i] ** w
                s2 += oi[i]
            if s2 > 0:
                for i in range(c):
                    oi[i] /= s2
    return om, oi


@njit(cache=True)
def _clean_counts(off, mz, it, floor, topk, power, ent_weight):
    n = len(off) - 1
    cnt = np.zeros(n, np.int64)
    for k in range(n):
        a = off[k]; b = off[k + 1]
        if b > a:
            cm, cp = _clean(mz[a:b], it[a:b], floor, topk, power, ent_weight)
            cnt[k] = len(cm)
    return cnt


@njit(cache=True, parallel=True)
def _clean_fill(off, mz, it, floor, topk, power, ent_weight, noff, omz, oit):
    n = len(off) - 1
    for k in prange(n):
        a = off[k]; b = off[k + 1]
        if b > a:
            cm, cp = _clean(mz[a:b], it[a:b], floor, topk, power, ent_weight)
            o = noff[k]
            for i in range(len(cm)):
                omz[o + i] = cm[i]
                oit[o + i] = cp[i]


def clean_batch(off, mz, it, floor, topk, power, ent_weight):
    """Clean a CSR batch of spectra. Returns (new_off int64, mz float32, it float32)."""
    mz = np.ascontiguousarray(mz, np.float32); it = np.ascontiguousarray(it, np.float32)
    off = np.ascontiguousarray(off, np.int64)
    cnt = _clean_counts_par(off, mz, it, floor, topk, power, ent_weight)
    noff = np.zeros(len(cnt) + 1, np.int64)
    np.cumsum(cnt, out=noff[1:])
    omz = np.empty(noff[-1], np.float32); oit = np.empty(noff[-1], np.float32)
    _clean_fill(off, mz, it, floor, topk, power, ent_weight, noff, omz, oit)
    return noff, omz, oit


@njit(cache=True, parallel=True)
def _clean_counts_par(off, mz, it, floor, topk, power, ent_weight):
    n = len(off) - 1
    cnt = np.zeros(n, np.int64)
    for k in prange(n):
        a = off[k]; b = off[k + 1]
        if b > a:
            cm, cp = _clean(mz[a:b], it[a:b], floor, topk, power, ent_weight)
            cnt[k] = len(cm)
    return cnt


@njit(cache=True, fastmath=True)
def entropy_sim(qmz, qp, cmz, cp, tol):
    i = 0; j = 0; n = len(qmz); m = len(cmz)
    SA = 0.0
    for x in range(n):
        if qp[x] > 0:
            SA -= qp[x] * np.log(qp[x])
    SB = 0.0
    for x in range(m):
        if cp[x] > 0:
            SB -= cp[x] * np.log(cp[x])
    SAB = 0.0; tot = 0.0
    buf = np.empty(n + m, np.float64); b = 0
    while i < n and j < m:
        d = qmz[i] - cmz[j]
        if d < -tol:
            buf[b] = qp[i]; i += 1; b += 1
        elif d > tol:
            buf[b] = cp[j]; j += 1; b += 1
        else:
            buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1
    while i < n:
        buf[b] = qp[i]; i += 1; b += 1
    while j < m:
        buf[b] = cp[j]; j += 1; b += 1
    for x in range(b):
        tot += buf[x]
    if tot <= 0:
        return 0.0
    for x in range(b):
        v = buf[x] / tot
        if v > 0:
            SAB -= v * np.log(v)
    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)


@njit(cache=True, fastmath=True)
def entropy_sim_shift(qmz, qp, cmz, cp, tol, shift):
    a = entropy_sim(qmz, qp, cmz, cp, tol)
    if shift > -0.001 and shift < 0.001:
        return a
    sm = np.empty(len(cmz), np.float32)
    for i in range(len(cmz)):
        sm[i] = cmz[i] + shift
    b = entropy_sim(qmz, qp, sm, cp, tol)
    return a if a > b else b


@njit(cache=True, fastmath=True, parallel=True)
def search_clean(qmz, qp, cand, off, allmz, allin, tol):
    """Entropy similarity of one cleaned query vs pre-cleaned library spectra `cand`."""
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a:
            continue
        out[k] = entropy_sim(qmz, qp, allmz[a:b], allin[a:b], tol)
    return out


@njit(cache=True, fastmath=True, parallel=True)
def search_shift_clean(qmz, qp, cand, off, allmz, allin, tol, shift):
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a:
            continue
        out[k] = entropy_sim_shift(qmz, qp, allmz[a:b], allin[a:b], tol, shift[k])
    return out


def clean(mz, it, cfg):
    return _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                  cfg.INT_FLOOR, cfg.MAX_PEAKS, cfg.INT_POWER, cfg.ENT_WEIGHT)


In [ ]:
%%writefile /kaggle/working/casmi/submit.py
"""Final list selection + submission writing/validation.

Rules (one invalid SMILES reportedly zeroes the whole submission on Kaggle):
  * every output SMILES must parse AND tautomer-canonicalize with RDKit 2026.03.3 (score_key not None)
  * dedupe by the metric key (tautomer-canonical InChIKey14) so no slot is wasted on a duplicate
  * <= 25 per molecule, no empty entries, no ';' inside a SMILES, fallback 'CCO' if nothing survives
"""
from __future__ import annotations

import time

import numpy as np
import polars as pl

from . import config
from .engine import Query
from .metric import parse_guesses, score_key

FALLBACK = 'CCO'


def select(smiles_ranked, n=25):
    """Walk a ranked SMILES list, keep valid & metric-unique entries. Returns (smiles, keys)."""
    out, keys, seen = [], [], set()
    for s in smiles_ranked:
        if not isinstance(s, str) or not s or ';' in s or ',' in s or s != s.strip():
            continue
        k = score_key(s)
        if k is None or k in seen:
            continue
        seen.add(k); out.append(s); keys.append(k)
        if len(out) >= n:
            break
    return out, keys


def validate_submission(df: pl.DataFrame, sample: pl.DataFrame, max_rank=25):
    assert df.columns[:2] == ['molecule_id', 'smiles'], df.columns
    assert len(df) == len(sample) and set(df['molecule_id']) == set(sample['molecule_id'])
    assert df['molecule_id'].is_unique().all(), 'duplicate molecule_id'
    assert df['smiles'].null_count() == 0
    n_bad = n_dup = 0
    for s in df['smiles'].to_list():
        g = parse_guesses(s)
        assert 1 <= len(g) <= max_rank, (len(g), s[:80])
        assert len(g) == len(s.split(';')), 'empty entry'
        ks = [score_key(x) for x in g]
        n_bad += sum(k is None for k in ks)
        n_dup += len(ks) - len(set(ks))
    assert n_bad == 0, f'{n_bad} unparseable SMILES'
    assert n_dup == 0, f'{n_dup} metric-duplicate SMILES'
    return True


def test_queries(path=config.TEST):
    from .chem import neutral_mass
    te = pl.read_parquet(path)
    nm = neutral_mass(te['precursor_mz'].to_numpy(), te['adduct'].to_numpy())
    te = te.with_columns(pl.Series('nm', nm))
    out = {}
    for mid, sub in te.group_by('molecule_id', maintain_order=True):
        mid = mid[0]
        nms = sub['nm'].to_numpy(); nms = nms[np.isfinite(nms)]
        if not len(nms):
            out[mid] = None; continue
        specs = [dict(mz=np.asarray(r['ms2_mzs'], np.float64), it=np.asarray(r['ms2_normalized_intensities'], np.float64),
                      prec=float(r['precursor_mz']), adduct=r['adduct'], instrument=r['instrument_type'],
                      ce=r['collision_energy_ev'], mode=1.0 if r['ionization_mode'] == 'positive' else -1.0)
                 for r in sub.iter_rows(named=True)]
        out[mid] = Query(specs, float(np.median(nms)))
    instr = te['instrument_type'].drop_nulls()
    pref = instr.mode().sort()[0] if len(instr) else None
    return out, pref


def predict_test(E, ranker, queries, cfg, log_every=50):
    """Rankers with a `prepare`/`features` pair (rows_v2.RankerV2) get a two-pass run: all candidate windows
    first, then one batched descriptor pass, then ranking. Other rankers score c['X'] (the 31 columns) directly."""
    rows, diag = {}, []
    t0 = time.time()
    two_pass = hasattr(ranker, 'prepare')
    wins = []
    for gi, (mid, q) in enumerate(queries.items()):
        c = z = None
        if q is not None:
            lib = E.lib_sim(q); an = E.analog_sim(q); z = E.logits(q)
            c = E.candidates(q, lib, an, z)
        if two_pass:
            wins.append((mid, q, c, z))
        else:
            rows[mid] = _rank_one(mid, q, c, ranker.predict(c['X']) if c is not None else None, cfg, diag)
        if gi % log_every == 0:
            print(f'  {gi}/{len(queries)}  {time.time()-t0:.0f}s', flush=True)
    if two_pass:
        ranker.prepare(E, [(q, c, z) for _, q, c, z in wins], cache=config.CACHE / 'prior_desc_test.pkl')
        for mid, q, c, z in wins:
            p = ranker.predict(ranker.features(E, q, c, z)) if c is not None else None
            rows[mid] = _rank_one(mid, q, c, p, cfg, diag)
        print(f'  ranked {len(wins)} ({time.time()-t0:.0f}s)', flush=True)
    return rows, diag


def _rank_one(mid, q, c, p, cfg, diag):
    smis = []
    if c is not None:
        order = np.argsort(-p)
        smis, _ = select([c["smiles"][i] for i in order], cfg.TOPN)
        diag.append(dict(molecule_id=mid, target=q.target, n_cand=len(c['smiles']), lib_max=float(c['lv'].max()),
                         top_analog=c['top_analog'], top_p=float(p[order[0]])))
    return smis or [FALLBACK]


def write_submission(rows: dict, out_path, sample_path=config.SAMPLE):
    sample = pl.read_csv(sample_path)
    df = pl.DataFrame(dict(molecule_id=sample['molecule_id'],
                           smiles=[';'.join(rows.get(m, [FALLBACK])) for m in sample['molecule_id'].to_list()]))
    validate_submission(df, sample)
    df.write_csv(out_path)
    return df


In [ ]:
%%writefile /kaggle/working/casmi/train_fpnet.py
"""Leak-free retraining of the spectrum -> fingerprint model (FPNet), prvsiyan's recipe.

Recipe (analog-propagation notebook, appendix / TRAIN_SCRIPT_SRC):
  loss   = BCE(z, f_true) + lam * softmax-CE over [positive + K=63 decoys from the same +-10 ppm pool window],
           score = (f.z - mean_c f.z) / sqrt(nbits), computed in fp32
  optim  = AdamW(lr 3e-4, wd 0.01, betas (0.9, 0.98)), warmup 2k, cosine to 2% of lr, bs 256, fp16 autocast, clip 1
  aug    = peak dropout U(0, 0.3), intensity log-normal sigma 0.25, m/z +-5 ppm
  merge  = with prob merge_p the input is 2-4 spectra of the same structure merged (fp_merged_*: 0.6, fp_single_*: 0)
  select = keep only the best checkpoint by validation hard-negative top-1 (the model overfits after ~12-20k steps)
Changes vs the notebook:
  * ALL structures of the fixed held-out list (holdout_v1.json: every metric key used by CV) are removed from the
    training spectra AND from the decoy pool, so CV on those molecules no longer leaks through the FP channel.
  * validation = a fixed random 1.5% of the remaining metric keys (never the CV molecules), deterministic negatives;
    the selection score is the mean of all-val and timsTOF-val hard-negative top-1.
  * timsTOF spectra (the test instrument) are over-sampled to `tims_frac` of every batch.
  * adduct / CE / instrument / polarity conditioning is the architecture's global token (unchanged, fpnet.FPNet).

Commands
  uv run python -m casmi.train_fpnet holdout            # writes src/casmi/holdout_v1.json (needs data/cache)
  uv run python -m casmi.train_fpnet prep --out DIR     # spec.npz + pool.npz for training (CPU, ~5 min)
  python -m casmi.train_fpnet train --data DIR --out fp_merged_h1.pt --merge_p 0.6     # GPU
"""
from __future__ import annotations

import argparse
import json
import math
import os
import pickle
import time
from pathlib import Path

import numpy as np
import torch
import torch.utils.data

HOLDOUT_FILE = Path(__file__).with_name('holdout_v1.json')


# ============================================================================================ held-out list
def make_holdout(n_extra_e180=500, val_frac=0.015, seed=1):
    """cv   = the exp000 CV molecules (250 np-examples = all of them, 250 enveda-180), copied from
              outputs/cv/exp000/feats.pkl (cv.select_holdout(L, 250, 250, seed=0) at the time of exp000);
       extra= n_extra_e180 more enveda-180 molecules (never shown to FPNet; available for a bigger CV);
       fpval= metric keys used only for FPNet checkpoint selection."""
    from . import config
    from .cv import select_holdout
    from .library import Library
    cfg = config.CFG()
    L = Library(cfg)
    feats = config.OUTPUTS / 'cv' / 'exp000' / 'feats.pkl'
    if feats.exists():   # the exp000 records verbatim (query spectra included) -> exp000 features stay comparable
        cv = [r['h'] for r in pickle.load(open(feats, 'rb'))]
        print(f'cv part = exp000 holdout from {feats}')
    else:
        cv = select_holdout(L, 250, 250, 0)
    used = {h['mkey'] for h in cv}
    extra = select_holdout(L, 0, n_extra_e180, seed, exclude_mkeys=used)
    used |= {h['mkey'] for h in extra}
    # every structure's metric key that is not held out -> sample the FPNet validation keys
    rng = np.random.default_rng(seed + 1)
    rest = sorted({k for k in L.s_mkey if k is not None and k not in used})
    fpval = sorted(rng.choice(rest, size=int(len(rest) * val_frac), replace=False).tolist())
    out = dict(version=1, note='cv = exp000 CV holdout; extra = more enveda-180 held out of FPNet training; '
                               'held-out mkeys = cv + extra (excluded from FPNet spectra and decoy pool); '
                               'fpval = FPNet checkpoint-selection keys',
               cv=cv, extra=extra, fpval=fpval)
    HOLDOUT_FILE.write_text(json.dumps(out))
    print(f'wrote {HOLDOUT_FILE}: cv={len(cv)} extra={len(extra)} fpval={len(fpval)}')


def load_holdout():
    return json.loads(HOLDOUT_FILE.read_text())


def held_mkeys(H=None):
    H = H or load_holdout()
    return {h['mkey'] for h in H['cv']} | {h['mkey'] for h in H['extra']}


# ============================================================================================ data prep (CPU)
def prep(out_dir, workers=8):
    import polars as pl
    import pyarrow.parquet as pq
    from multiprocessing import Pool as MPool

    from . import config
    from .fpnet import ADDUCT_IX, instr_family
    from .metric import score_keys_parallel
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    H = load_holdout()
    held = held_mkeys(H); fpval = set(H['fpval'])
    st = pl.read_parquet(config.CACHE / 'structs.parquet')
    s_key = st['inchikey14'].to_numpy(); s_mkey = st['mkey'].to_numpy()
    s_held = np.array([k in held for k in s_mkey]); s_val = np.array([k in fpval for k in s_mkey])
    print(f'structures {len(st):,}: held {s_held.sum()} val {s_val.sum()}', flush=True)

    # ---- decoy/target pool = COCONUT U train structures (same as candidates.Pool), minus held-out metric keys
    cm = pickle.load(open(config.COCO_DIR / 'coco_meta.pkl', 'rb'))
    tm = pl.read_parquet(config.CACHE / 'pool_train_meta.parquet')
    fp = np.vstack([np.load(config.COCO_DIR / 'coco_fp.npy'), np.load(config.CACHE / 'pool_train_fp.npy')])
    mass = np.concatenate([np.load(config.COCO_DIR / 'coco_mass.npy'), tm['mass'].to_numpy()])
    keys = np.concatenate([np.asarray(cm['keys'], dtype=object), tm['key'].to_numpy().astype(object)])
    smis = np.concatenate([np.asarray(cm['smiles'], dtype=object), tm['smiles'].to_numpy().astype(object)])
    good = np.isfinite(mass)
    held_keys14 = set(s_key[s_held].tolist())
    drop = np.array([k in held_keys14 for k in keys])
    # COCONUT tautomer variants of held-out molecules: metric-key the COCONUT entries near a held-out mass
    hm = np.sort(mass[drop & good])
    near = np.zeros(len(mass), bool)
    if len(hm):
        j = np.clip(np.searchsorted(hm, mass), 1, len(hm) - 1)
        d = np.minimum(np.abs(mass - hm[j - 1]), np.abs(mass - hm[j]))
        near = good & ~drop & (d <= 1e-5 * mass)
    idx = np.where(near)[0]
    mk = score_keys_parallel(smis[idx].tolist(), workers=workers)
    extra_drop = idx[np.array([k in held for k in mk], bool)] if len(idx) else idx
    drop[extra_drop] = True
    print(f'pool: dropped {drop.sum()} held-out entries ({len(extra_drop)} via metric key among {len(idx)} '
          f'same-mass) ({time.time()-t0:.0f}s)', flush=True)
    keep = good & ~drop
    fp, mass, keys = fp[keep], mass[keep], keys[keep]
    o = np.argsort(mass, kind='mergesort'); fp, mass, keys = fp[o], mass[o], keys[o]
    k2p = {k: i for i, k in enumerate(keys)}
    s2p = np.array([k2p.get(k, -1) for k in s_key], np.int64)
    s2p[s_held] = -1
    np.save(out / 'pool_fp.npy', np.ascontiguousarray(fp)); np.save(out / 'pool_mass.npy', mass)
    print(f'pool {len(mass):,} x {fp.shape[1]} bytes; train structures with fp {int((s2p >= 0).sum()):,}', flush=True)

    # ---- spectra (row order == library cache order)
    meta = pl.read_parquet(config.CACHE / 'lib_meta.parquet', columns=['sid', 'instrument', 'adduct', 'mode', 'ce'])
    sid = meta['sid'].to_numpy()
    pf = pq.ParquetFile(config.TRAIN)
    offs, mzs, its, keep_rows = [np.zeros(1, np.int64)], [], [], []
    base, row0 = 0, 0
    for rg in range(pf.num_row_groups):
        t = pf.read_row_group(rg, columns=['ms2_mzs', 'ms2_normalized_intensities', 'precursor_mz'])
        n = t.num_rows
        rows = np.arange(row0, row0 + n); row0 += n
        ok = s2p[sid[rows]] >= 0
        args = [(r['ms2_mzs'], r['ms2_normalized_intensities'], r['precursor_mz'])
                for r, k in zip(t.to_pylist(), ok) if k]
        with MPool(workers) as p:
            res = p.starmap(_prep_one, args, chunksize=2000)
        lens = np.array([len(a) for a, _ in res], np.int64)
        good_r = lens > 0
        kr = rows[ok][good_r]
        keep_rows.append(kr)
        mzs.append(np.concatenate([a for a, _ in res if len(a)]) if good_r.any() else np.zeros(0, np.float32))
        its.append(np.concatenate([b for _, b in res if len(b)]).astype(np.float16) if good_r.any() else np.zeros(0, np.float16))
        offs.append(base + np.cumsum(lens[good_r])); base += int(lens.sum())
        print(f'  rg {rg+1}/{pf.num_row_groups}: kept {len(kr):,}/{n:,} ({time.time()-t0:.0f}s)', flush=True)
    rows = np.concatenate(keep_rows)
    m = meta[rows]
    prec = pl.read_parquet(config.TRAIN, columns=['precursor_mz'])['precursor_mz'].to_numpy()[rows]
    ins_u = {}
    ins = np.array([ins_u.setdefault(x, instr_family(x)) for x in m['instrument'].to_list()], np.int8)
    ad = np.array([ADDUCT_IX.get(a, ADDUCT_IX['<unk>']) for a in m['adduct'].to_list()], np.int8)
    mode = np.where(m['mode'].to_numpy() == 'positive', 1.0, -1.0).astype(np.float32)
    ce = m['ce'].to_numpy().astype(np.float32)
    ssid = sid[rows]
    np.savez(out / 'spec.npz', off=np.concatenate(offs), mz=np.concatenate(mzs), it=np.concatenate(its),
             prec=prec.astype(np.float32), ad=ad, ins=ins, ce=ce, mode=mode, pidx=s2p[ssid].astype(np.int32),
             is_val=s_val[ssid], row=rows.astype(np.int64))
    print(f'spec.npz: {len(rows):,} spectra ({(ins == 0).sum():,} timsTOF, {s_val[ssid].sum():,} val), '
          f'{base:,} peaks ({time.time()-t0:.0f}s)', flush=True)
    (out / 'prep_meta.json').write_text(json.dumps(dict(nbits=int(cm['nbits']), n_spec=int(len(rows)),
                                                         n_pool=int(len(mass)), n_held=len(held))))


def _prep_one(mz, it, prec):
    from .fpnet import prep_peaks
    return prep_peaks(mz, it, float(prec))


# ============================================================================================ training (GPU)
class Spec:
    def __init__(self, d):
        z = np.load(Path(d) / 'spec.npz')
        for k in ('off', 'mz', 'it', 'prec', 'ad', 'ins', 'ce', 'mode', 'pidx', 'is_val'):
            setattr(self, k, z[k])
        self.mass = np.load(Path(d) / 'pool_mass.npy')
        o = np.argsort(self.pidx, kind='mergesort')
        self.g_order = o
        self.g_bounds = np.searchsorted(self.pidx[o], np.arange(len(self.mass) + 1))

    def peers(self, i, rng, kmax=4):
        p = self.pidx[i]; a, b = self.g_bounds[p], self.g_bounds[p + 1]
        if b - a <= 1: return [i]
        k = int(rng.integers(2, min(kmax, b - a) + 1))
        pick = rng.choice(self.g_order[a:b], size=k, replace=False)
        if i not in pick: pick = np.concatenate([[i], pick[:-1]])
        return list(pick)

    def merged(self, idxs, maxlen):
        mz = np.concatenate([self.mz[self.off[j]:min(self.off[j] + maxlen, self.off[j + 1])] for j in idxs])
        it = np.concatenate([self.it[self.off[j]:min(self.off[j] + maxlen, self.off[j + 1])] for j in idxs]).astype(np.float32)
        o = np.argsort(mz, kind='stable'); mz, it = mz[o], it[o]
        # near-duplicate m/z: drop the weaker of each close adjacent pair (== the notebook's sequential loop)
        close = np.diff(mz) < 0.005
        keep = np.ones(len(mz), bool)
        j = np.where(close)[0] + 1
        w = it[j] >= it[j - 1]
        keep[j[w] - 1] = False; keep[j[~w]] = False
        mz, it = mz[keep], it[keep]
        if len(mz) > maxlen:
            top = np.sort(np.argsort(-it)[:maxlen]); mz, it = mz[top], it[top]
        return mz, it

    def negs(self, pos, K, rng, ppm=10.0):
        m = self.mass[pos]; tol = m * ppm / 1e6
        lo = np.searchsorted(self.mass, m - tol, 'left'); hi = np.searchsorted(self.mass, m + tol, 'right')
        n = hi - lo
        c = lo[:, None] + np.floor(rng.random((len(pos), K)) * np.maximum(n, 1)[:, None]).astype(np.int64)
        bad = c == pos[:, None]
        c = np.where(bad, np.where(c + 1 < hi[:, None], c + 1, lo[:, None]), c)
        few = n <= 1
        if few.any(): c[few] = rng.integers(0, len(self.mass), (few.sum(), K))
        return c

    def batch(self, ids, K, rng, aug, merge_p, max_peaks=128):
        B = len(ids)
        lens = np.minimum(self.off[ids + 1] - self.off[ids], max_peaks); N = max(int(lens.max()), 1)
        mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)
        for r, (i, l) in enumerate(zip(ids, lens)):
            a = self.off[i]; mz[r, :l] = self.mz[a:a + l]; it[r, :l] = self.it[a:a + l]; pad[r, :l] = False
        if merge_p > 0:
            for r, i in enumerate(ids):
                if rng.random() >= merge_p: continue
                pk = self.peers(i, rng)
                if len(pk) < 2: continue
                m2, i2 = self.merged(pk, N)
                mz[r] = 0; it[r] = 0; pad[r] = True
                mz[r, :len(m2)] = m2; it[r, :len(m2)] = i2; pad[r, :len(m2)] = False
        if aug:
            keep = rng.random((B, N)) > rng.uniform(0.0, 0.30, size=(B, 1))
            pad = pad | ~keep
            it = it * np.exp(rng.normal(0.0, 0.25, size=(B, N))).astype(np.float32)
            mz = mz * (1.0 + rng.normal(0.0, 5e-6, size=(B, N))).astype(np.float32)
            allpad = pad.all(1)
            if allpad.any(): pad[allpad, 0] = False
        ce = np.where(self.ce[ids] < 0, 25.0, self.ce[ids]).astype(np.float32)
        pos = self.pidx[ids].astype(np.int64)
        cand = np.concatenate([pos[:, None], self.negs(pos, K, rng)], 1) if K > 0 else pos[:, None]
        return (mz, it, pad, self.prec[ids].astype(np.float32), self.ad[ids].astype(np.int64),
                self.ins[ids].astype(np.int64), ce, self.mode[ids].astype(np.float32), cand)


def _train_stream(data_dir, seed, bs, K, merge_p, tims_frac):
    import torch
    wi = torch.utils.data.get_worker_info()
    wid = wi.id if wi else 0
    D = Spec(data_dir)
    rng = np.random.default_rng([seed, wid])
    tr = np.where(~D.is_val)[0]
    tims = tr[D.ins[tr] == 0]; other = tr[D.ins[tr] != 0]
    while True:
        nt = rng.binomial(bs, tims_frac) if tims_frac >= 0 else None
        if nt is None:
            ids = rng.choice(tr, size=bs, replace=False)
        else:
            ids = np.concatenate([rng.choice(tims, nt, replace=False), rng.choice(other, bs - nt, replace=False)])
        yield D.batch(ids, K, rng, True, merge_p)


class StreamDS(torch.utils.data.IterableDataset):
    def __init__(self, *args): self.args = args

    def __iter__(self): return _train_stream(*self.args)


def train(a):
    import torch
    import torch.nn.functional as F
    from torch.utils.data import DataLoader

    from .fpnet import FPNet

    dev = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
    meta = json.loads((Path(a.data) / 'prep_meta.json').read_text()); nbits = meta['nbits']
    PFP = torch.from_numpy(np.load(Path(a.data) / 'pool_fp.npy')).to(dev)       # packed (n_pool, ceil(nbits/8))
    shifts = torch.arange(7, -1, -1, device=dev, dtype=torch.uint8)

    def unpack(idx):  # (B, C) -> (B, C, nbits) float, np.packbits big-endian order
        p = PFP[idx]
        return ((p.unsqueeze(-1) >> shifts) & 1).flatten(-2)[..., :nbits].float()

    D = Spec(a.data)
    va_all = np.where(D.is_val)[0]
    vr = np.random.default_rng(12345)
    va_t = va_all[D.ins[va_all] == 0]; va_o = va_all[D.ins[va_all] != 0]
    VAL = dict(all=np.sort(vr.choice(va_all, min(a.n_val, len(va_all)), replace=False)),
               tims=np.sort(vr.choice(va_t, min(a.n_val, len(va_t)), replace=False)))
    print(f'device {dev}; nbits {nbits}; spectra {len(D.pidx):,} (val {len(va_all):,}, val timsTOF {len(va_t):,}); '
          f'pool {len(D.mass):,}', flush=True)

    torch.manual_seed(a.seed)
    model = FPNet(nbits, d=a.d, layers=a.layers).to(dev)
    print('params %.1fM' % (sum(p.numel() for p in model.parameters()) / 1e6), flush=True)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=0.01, betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler('cuda') if dev == 'cuda' else None

    def lr_at(s):
        if s < a.warm: return a.lr * s / max(1, a.warm)
        p = (s - a.warm) / max(1, a.steps - a.warm)
        return a.lr * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * min(p, 1.0))))

    T = lambda x: torch.as_tensor(x, device=dev)

    def fwd(b):
        inp = tuple(T(x) for x in b[:8])
        ctx = torch.autocast('cuda', dtype=torch.float16) if dev == 'cuda' else torch.autocast('cpu', enabled=False)
        with ctx:
            z = model(*inp)
        z = z.float()
        cand = T(b[8])
        ypos = unpack(cand[:, 0])
        FPc = unpack(cand)
        raw = torch.bmm(FPc, z.unsqueeze(-1)).squeeze(-1)
        sc = (raw - raw.mean(1, keepdim=True)) / math.sqrt(nbits)
        return z, ypos, sc

    def evaluate():
        model.eval(); out = {}
        with torch.no_grad():
            for name, ids in VAL.items():
                rng = np.random.default_rng(777)          # identical negatives / merges every evaluation
                acc, bce = [], []
                for s in range(0, len(ids), 128):
                    b = D.batch(ids[s:s + 128], a.K, rng, False, a.merge_p)
                    z, ypos, sc = fwd(b)
                    bce.append(F.binary_cross_entropy_with_logits(z, ypos).item())
                    acc.append((sc.argmax(1) == 0).float().sum().item())
                out[name] = (float(np.sum(acc) / len(ids)), float(np.mean(bce)))
        model.train()
        return out

    def save(step, path, val):
        torch.save({'model': model.state_dict(), 'step': step, 'nbits': nbits, 'd': a.d, 'layers': a.layers,
                    'val': val, 'args': vars(a)}, path)

    dl = DataLoader(StreamDS(a.data, a.seed, a.bs, a.K, a.merge_p, a.tims_frac), batch_size=None, num_workers=a.workers, persistent_workers=a.workers > 0,
                    prefetch_factor=4 if a.workers > 0 else None)
    it_dl = iter(dl)
    t0 = time.time(); rb = rc = racc = 0.0; nr = 0; best = -1.0; hist = []
    for step in range(a.steps):
        for g in opt.param_groups: g['lr'] = lr_at(step)
        b = next(it_dl)
        b = tuple(x.numpy() if hasattr(x, 'numpy') else x for x in b)
        z, ypos, sc = fwd(b)
        lb = F.binary_cross_entropy_with_logits(z, ypos)
        lc = F.cross_entropy(sc, torch.zeros(len(sc), dtype=torch.long, device=dev))
        loss = lb + a.lam * lc
        opt.zero_grad(set_to_none=True)
        if not torch.isfinite(loss):
            print(f'  non-finite loss at step {step}; skipping', flush=True); continue
        if scaler is not None:
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
        else:
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        rb += lb.item(); rc += lc.item(); racc += (sc.argmax(1) == 0).float().mean().item(); nr += 1
        if step % 200 == 0:
            el = time.time() - t0
            print(f'step {step} bce {rb/nr:.4f} ctr {rc/nr:.4f} top1 {racc/nr:.3f} lr {lr_at(step):.2e} '
                  f'{el:.0f}s {(step+1)/max(el,1):.2f} it/s', flush=True)
            rb = rc = racc = 0.0; nr = 0
        last = (step + 1 == a.steps) or (time.time() - t0) / 60 > a.max_minutes
        if (step + 1) % a.val_every == 0 or last:
            v = evaluate()
            score = 0.5 * (v['all'][0] + v['tims'][0])
            hist.append(dict(step=step + 1, score=score, **{k: x[0] for k, x in v.items()}))
            print(f'  VAL step {step+1} top1 all {v["all"][0]:.4f} tims {v["tims"][0]:.4f} bce {v["all"][1]:.4f} '
                  f'score {score:.4f}' + ('  <- best' if score > best else ''), flush=True)
            if score > best:
                best = score; save(step + 1, a.out, v)
            Path(a.out).with_suffix('.hist.json').write_text(json.dumps(hist, indent=0))
        if last and (time.time() - t0) / 60 > a.max_minutes:
            print('time budget reached', flush=True); break
    print(f'done. best score {best:.4f}; {(time.time()-t0)/60:.1f} min', flush=True)


def main(argv=None):
    ap = argparse.ArgumentParser('train_fpnet')
    sub = ap.add_subparsers(dest='cmd', required=True)
    h = sub.add_parser('holdout'); h.add_argument('--n-extra-e180', type=int, default=500)
    p = sub.add_parser('prep'); p.add_argument('--out', required=True); p.add_argument('--workers', type=int, default=8)
    t = sub.add_parser('train')
    t.add_argument('--data', required=True); t.add_argument('--out', default='fp_merged_h1.pt')
    t.add_argument('--steps', type=int, default=30000); t.add_argument('--bs', type=int, default=256)
    t.add_argument('--lr', type=float, default=3e-4); t.add_argument('--d', type=int, default=512)
    t.add_argument('--layers', type=int, default=6); t.add_argument('--K', type=int, default=63)
    t.add_argument('--lam', type=float, default=1.0); t.add_argument('--warm', type=int, default=2000)
    t.add_argument('--val_every', type=int, default=1000); t.add_argument('--n_val', type=int, default=4096)
    t.add_argument('--max_minutes', type=float, default=1e9); t.add_argument('--seed', type=int, default=7)
    t.add_argument('--merge_p', type=float, default=0.6)
    t.add_argument('--tims_frac', type=float, default=0.6, help='-1 = uniform sampling (the notebook)')
    t.add_argument('--workers', type=int, default=3)
    a = ap.parse_args(argv)
    if a.cmd == 'holdout':
        make_holdout(a.n_extra_e180)
    elif a.cmd == 'prep':
        prep(a.out, a.workers)
    else:
        train(a)


if __name__ == '__main__':
    main()


In [ ]:
import glob, os, subprocess, sys
from pathlib import Path

# Offline RDKit pinned to the metric's version.
py = f"cp{sys.version_info.major}{sys.version_info.minor}"
wheel = glob.glob(f"/kaggle/input/**/rdkit-2026.3.3-{py}-*.whl", recursive=True)[0]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--no-index", wheel], check=True)

# Lay the inputs out the way casmi.config expects: data/{train,test}.parquet and data/ext/<dataset>.
data = Path("/kaggle/working/data"); (data / "ext").mkdir(parents=True, exist_ok=True)
comp = Path(sorted(p for p in glob.glob("/kaggle/input/**/test.parquet", recursive=True) if "enveda-CASMI26-molecule-id-mass-spectra" in p)[0]).parent
for f in ["train.parquet", "test.parquet", "sample_submission.csv"]:
    (data / f).unlink(missing_ok=True); (data / f).symlink_to(comp / f)
for name in ['casmi26-fp-models-v2', 'casmi26-ranker-features', 'chebi-lipidmaps-casmi26', 'coconut-casmi26-candidates']:
    src = [p for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if os.path.isdir(p)][0]
    (data / "ext" / name).unlink(missing_ok=True); (data / "ext" / name).symlink_to(src)
os.environ["CASMI_DATA"] = str(data)
os.environ["CASMI_OUTPUTS"] = "/kaggle/working/outputs"
# Caches are ~1GB; keep them out of /kaggle/working so they aren't saved as notebook output.
os.environ["CASMI_CACHE"] = "/tmp/casmi_cache"
# CV feature rows (for --ranker cv) must sit at outputs/cv/<tag>/feats.pkl.
feats = glob.glob("/kaggle/input/**/casmi26-cv-feats/**/feats.pkl", recursive=True)
if feats:
    dst = Path("/kaggle/working/outputs/cv") / 'exp000' / "feats.pkl"
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.unlink(missing_ok=True); dst.symlink_to(feats[0])
    print("cv feats:", feats[0])
sys.path.insert(0, "/kaggle/working")
print(comp, sorted(os.listdir(data / "ext")))
FP_DIR = None
if 'mins00/casmi26-fp-models-h1' != "None":
    FP_DIR = [p for p in glob.glob("/kaggle/input/**/casmi26-fp-models-h1", recursive=True) if os.path.isdir(p)][0]
    print("fp weights:", FP_DIR, sorted(os.listdir(FP_DIR)))
ROWS_DIR = None
if 'v2' == "v2":
    ROWS_DIR = str(Path(glob.glob("/kaggle/input/**/casmi26-ranker-rows-v2/**/rows.npz", recursive=True)[0]).parent)
    print("v2 rows:", ROWS_DIR, sorted(os.listdir(ROWS_DIR)))


In [ ]:
import time; t0 = time.time()
from casmi.cli import main
main(['build'])
print(f'build {time.time()-t0:.0f}s')

In [ ]:
RANKER, CV_TAG = 'v2', 'exp000'
t0 = time.time()
main(['predict', '--ranker', RANKER, '--cv-tag', CV_TAG, '--device', 'cuda' if __import__('torch').cuda.is_available() else 'cpu',
      '--out', '/kaggle/working/submission.csv'] + (['--fp-dir', FP_DIR] if FP_DIR else [])
     + (['--ranker-rows', ROWS_DIR, '--v2-model', 'blend', '--v2-cols', 'base'] if ROWS_DIR else []))
print(f'predict {time.time()-t0:.0f}s')

In [ ]:
import polars as pl
from rdkit import Chem
sub = pl.read_csv('/kaggle/working/submission.csv')
bad = [s for row in sub['smiles'] for s in row.split(';') if Chem.MolFromSmiles(s) is None]
assert not bad, bad[:5]
print(sub.shape, 'all SMILES parse')